# Explainable FDI detection, localization and trust-gated alarms on HAI 21.03

Complete, runnable pipeline for the paper *Where Does the Explanation Point?* (Yogalakshmi S, Suresh A).

Run the cells from top to bottom (**Runtime → Run all**). Total time: about 10 minutes for HAI 21.03 (sections 1–7) and 12–15 minutes more for HAI 22.04 (section 8) on a standard Colab CPU. No GPU needed.

## 1. Install libraries and download the HAI 21.03 dataset (~180 MB)

In [ ]:
!pip -q install numpy pandas scikit-learn scipy matplotlib
import os
if not os.path.isdir('hai/hai-21.03'):
    !git clone -q --depth 1 --filter=blob:none --no-checkout https://github.com/icsdataset/hai
    !cd hai && git checkout HEAD -- hai-21.03
!ls -lh hai/hai-21.03

## 2. Write the pipeline script

Detectors (PCA, MLP-AE, IForest, Z-max), attributions (residual, Z-mean, exact group Shapley), stealth tiers and the trust gate. Every number in the paper is written by this script.

In [ ]:
%%writefile fdi_pipeline.py
"""
Explainable FDI detection, process-level localization, stealth breakdown and
trust-gated alarms on HAI 21.03.

Usage:
    python fdi_pipeline.py --data /path/to/hai/hai-21.03 --out results

Everything reported in the paper is written by this script to --out
(CSV/JSON tables + PNG figures). Nothing is typed in by hand.
"""
import argparse, glob, json, os, time
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import average_precision_score

GROUPS = ["P1", "P2", "P3", "P4"]
SEEDS = [42, 123, 2026, 7, 99]
SMOOTH = 60            # trailing moving-average window for the anomaly score (s)
GRACE = 60             # seconds after an event end in which an alarm still counts
CAL_FRAC = 0.20        # chronological tail of train3 held out for threshold calibration
THR_Q = 0.999          # threshold = this quantile of calibration scores
GATE_K = 10            # seconds of evidence needed by duration filter / trust gate
NOISE_SD = 0.05        # perturbation (z-units) used for explanation-stability check
NOISE_REPS = 5
GATE_QS = [0.99, 0.995, 0.999]
GATE_SDS = [0.05, 0.2, 0.5]


# ----------------------------------------------------------------- data
def load(data_dir):
    tr = [pd.read_csv(f) for f in sorted(glob.glob(f"{data_dir}/train*.csv.gz"))]
    te_files = sorted(glob.glob(f"{data_dir}/test*.csv.gz"))
    te = []
    for i, f in enumerate(te_files):
        d = pd.read_csv(f)
        d["file"] = i
        te.append(d)
    return tr, pd.concat(te, ignore_index=True)


def prepare(tr, te):
    sens = [c for c in tr[0].columns if c[:2] in GROUPS]
    trall = pd.concat(tr, ignore_index=True)
    # drop sensors that never change in training (one distinct value). Checking
    # nunique instead of std == 0 avoids floating-point rounding differences
    # between pandas/NumPy versions (e.g. P2_MSD = 763.19324 everywhere).
    feats = [c for c in sens if trall[c].nunique(dropna=True) > 1]
    # chronological calibration tail of the last training file
    n3 = len(tr[-1]); cut = int(n3 * (1 - CAL_FRAC))
    fit_df = pd.concat(tr[:-1] + [tr[-1].iloc[:cut]], ignore_index=True)
    cal_df = tr[-1].iloc[cut:].reset_index(drop=True)
    mu, sig = fit_df[feats].mean(), fit_df[feats].std()
    sig = sig.where(sig > 1e-8, 1.0)                        # guard against near-zero spread
    z = lambda d: ((d[feats] - mu) / sig).to_numpy(np.float32)
    gidx = {g: np.array([i for i, c in enumerate(feats) if c.startswith(g)]) for g in GROUPS}
    base = np.median(z(fit_df), axis=0).astype(np.float32)
    return feats, gidx, z(fit_df), z(cal_df), z(te), base


def events(te):
    a = te["attack"].to_numpy()
    s = np.flatnonzero(np.diff(np.r_[0, a]) == 1)
    e = np.flatnonzero(np.diff(np.r_[a, 0]) == -1)
    out = []
    for k, (i, j) in enumerate(zip(s, e)):
        seg = te.iloc[i:j + 1]
        tg = [g for g in ["P1", "P2", "P3"] if seg[f"attack_{g}"].max() > 0]
        out.append(dict(event=k, start=int(i), end=int(j), dur=int(j - i + 1),
                        file=int(seg["file"].iloc[0]), targets="+".join(tg),
                        single=len(tg) == 1))
    return pd.DataFrame(out)


# ------------------------------------------------------------ detectors
class Detector:
    """score(X) -> per-row anomaly score; resid(X) -> per-feature residual (or None)."""
    name = ""
    def fit(self, X): ...
    def score(self, X): ...
    def resid(self, X): return None


class PCARecon(Detector):
    name = "PCA"
    def __init__(self, seed=0): self.seed = seed
    def fit(self, X):
        self.m = PCA(n_components=0.99, random_state=self.seed).fit(X); return self
    def resid(self, X): return X - self.m.inverse_transform(self.m.transform(X))
    def score(self, X): return np.mean(self.resid(X) ** 2, axis=1)


class MLPAE(Detector):
    name = "MLP-AE"
    def __init__(self, seed=0): self.seed = seed
    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(len(X), size=min(120_000, len(X)), replace=False)
        self.m = MLPRegressor(hidden_layer_sizes=(64, 16, 64), activation="relu",
                              solver="adam", batch_size=512, max_iter=80,
                              early_stopping=True, n_iter_no_change=5,
                              random_state=self.seed).fit(X[idx], X[idx])
        return self
    def resid(self, X): return X - self.m.predict(X)
    def score(self, X): return np.mean(self.resid(X) ** 2, axis=1)


class IForest(Detector):
    name = "IForest"
    def __init__(self, seed=0): self.seed = seed
    def fit(self, X):
        self.m = IsolationForest(n_estimators=200, max_samples=4096,
                                 random_state=self.seed, n_jobs=-1).fit(X[::5]); return self
    def score(self, X): return -self.m.score_samples(X)


class ZMax(Detector):
    """Model-free baseline: largest absolute z-score of any sensor."""
    name = "Z-max"
    def __init__(self, seed=0): pass
    def fit(self, X): return self
    def resid(self, X): return X
    def score(self, X): return np.max(np.abs(X), axis=1)


def smooth(s, files):
    out = np.empty_like(s, dtype=np.float64)
    for f in np.unique(files):
        m = files == f
        out[m] = pd.Series(s[m]).rolling(SMOOTH, min_periods=1).mean().to_numpy()
    return out


# ---------------------------------------------------------- attribution
def attr_residual(det, X, gidx, cal_resid_sd):
    """Process attribution from per-feature residuals, normalised by their normal spread."""
    r = det.resid(X)
    if r is None:
        return None
    r = (r / cal_resid_sd) ** 2
    return np.stack([r[:, gidx[g]].mean(axis=1) for g in GROUPS], axis=1)


def attr_group_shapley(det, X, gidx, base):
    """Exact Shapley values of the anomaly score over the 4 process groups.
    Absent groups are replaced by the training median (16 coalitions)."""
    n, G = len(X), len(GROUPS)
    from itertools import combinations
    from math import factorial
    val = {}
    for k in range(G + 1):
        for S in combinations(range(G), k):
            Xs = np.repeat(base[None, :], n, axis=0)
            for g in S:
                Xs[:, gidx[GROUPS[g]]] = X[:, gidx[GROUPS[g]]]
            val[S] = det.score(Xs)
    phi = np.zeros((n, G))
    for g in range(G):
        others = [h for h in range(G) if h != g]
        for k in range(G):
            w = factorial(k) * factorial(G - k - 1) / factorial(G)
            for S in combinations(others, k):
                phi[:, g] += w * (val[tuple(sorted(S + (g,)))] - val[S])
    return phi


def attr_z(X, gidx):
    """Model-free attribution: mean |z| per process."""
    return np.stack([np.abs(X[:, gidx[g]]).mean(axis=1) for g in GROUPS], axis=1)


# --------------------------------------------------------------- metrics
def alarm_segments(alarm, files):
    segs = []
    for f in np.unique(files):
        idx = np.flatnonzero(files == f)
        a = alarm[idx]
        s = np.flatnonzero(np.diff(np.r_[0, a.astype(int)]) == 1)
        e = np.flatnonzero(np.diff(np.r_[a.astype(int), 0]) == -1)
        segs += [(idx[i], idx[j]) for i, j in zip(s, e)]
    return segs


def evaluate(alarm, y, files, ev, score_s=None):
    tp = int((alarm & (y == 1)).sum()); fp = int((alarm & (y == 0)).sum())
    fn = int((~alarm & (y == 1)).sum())
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    F1 = 2 * P * R / max(P + R, 1e-12)
    near = np.zeros_like(y, bool)
    det, delay = [], []
    for _, e in ev.iterrows():
        lo, hi = e.start, min(e.end + GRACE, len(y) - 1)
        near[lo:hi + 1] = True
        hit = np.flatnonzero(alarm[lo:hi + 1])
        det.append(len(hit) > 0); delay.append(hit[0] if len(hit) else np.nan)
    segs = alarm_segments(alarm, files)
    fa = sum(1 for s, t in segs if not near[s:t + 1].any())
    normal_hours = (~near).sum() / 3600
    return dict(precision=P, recall=R, f1=F1,
                auc_pr=average_precision_score(y, score_s) if score_s is not None else np.nan,
                event_recall=float(np.mean(det)),
                median_delay_s=float(np.nanmedian(delay)),
                false_alarms_per_hour=fa / normal_hours), np.array(det)


def duration_filter(alarm, files, k):
    out = np.zeros_like(alarm)
    for s, t in alarm_segments(alarm, files):
        if t - s + 1 >= k:
            out[s + k - 1:t + 1] = True      # raised once k seconds of evidence accumulate
    return out


def trust_gate(alarm, stable, files, k):
    """Raise an alarm segment only after k seconds whose explanation is stable."""
    out = np.zeros_like(alarm)
    for s, t in alarm_segments(alarm, files):
        c = np.cumsum(stable[s:t + 1])
        hit = np.flatnonzero(c >= k)
        if len(hit):
            out[s + hit[0]:t + 1] = True
    return out


def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results")
    ap.add_argument("--seeds", type=int, default=len(SEEDS))
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()

    tr, te = load(a.data)
    feats, gidx, Xfit, Xcal, Xte, base = prepare(tr, te)
    ev = events(te)
    y = te["attack"].to_numpy().astype(int); files = te["file"].to_numpy()
    in_attack = y == 1

    # stealth magnitude of each real event: 95th pct of max |z| over the target process(es)
    mag = []
    for _, e in ev.iterrows():
        cols = np.concatenate([gidx[g] for g in e.targets.split("+")])
        mag.append(float(np.percentile(np.abs(Xte[e.start:e.end + 1][:, cols]).max(axis=1), 95)))
    ev["magnitude_z"] = mag
    ev["stealth_tier"] = pd.qcut(ev["magnitude_z"], 3, labels=["high stealth", "medium", "low stealth"])
    ev.to_csv(f"{a.out}/events.csv", index=False)

    audit = dict(features_used=len(feats), dropped_constant=79 - len(feats),
                 group_sizes={g: int(len(gidx[g])) for g in GROUPS},
                 fit_rows=len(Xfit), cal_rows=len(Xcal), test_rows=len(Xte),
                 attack_rows=int(y.sum()), events=len(ev),
                 single_target_events=int(ev.single.sum()),
                 single_target_counts=ev[ev.single].targets.value_counts().to_dict(),
                 majority_class_hit_rate=float(ev[ev.single].targets.value_counts().iloc[0] / ev.single.sum()))
    json.dump(audit, open(f"{a.out}/dataset_audit.json", "w"), indent=2)

    det_rows, loc_rows, tier_rows, gate_rows, perev_rows, curve_rows = [], [], [], [], [], []
    for D in [PCARecon, MLPAE, IForest, ZMax]:
        seeds = SEEDS[:a.seeds] if D in (MLPAE, IForest) else SEEDS[:1]
        for seed in seeds:
            det = D(seed).fit(Xfit)
            cal_s = smooth(det.score(Xcal), np.zeros(len(Xcal)))
            thr = float(np.quantile(cal_s, THR_Q))
            raw = det.score(Xte); s = smooth(raw, files)
            alarm = s > thr
            m, detected = evaluate(alarm, y, files, ev, s)
            det_rows.append(dict(detector=D.name, seed=seed, threshold=thr, **m))

            # ---- localization on alarm points inside detected events
            cal_r = det.resid(Xcal)
            cal_sd = (np.std(cal_r, axis=0) + 1e-6) if cal_r is not None else None
            methods = {"Z-mean": lambda X: attr_z(X, gidx),
                       "GroupShapley": lambda X: attr_group_shapley(det, X, gidx, base)}
            if cal_sd is not None:
                methods["Residual"] = lambda X: attr_residual(det, X, gidx, cal_sd)
            for mname, fn in methods.items():
                hits = []
                for k, e in ev.iterrows():
                    allidx = np.arange(e.start, e.end + 1)
                    top_or = GROUPS[int(np.argmax(fn(Xte[allidx]).mean(axis=0)))]
                    hit_or = float(top_or in e.targets.split("+"))
                    idx = allidx[alarm[allidx]]
                    if len(idx) == 0:
                        hits.append(np.nan); top = None; hit = np.nan
                    else:
                        A = fn(Xte[idx]).mean(axis=0)
                        top = GROUPS[int(np.argmax(A))]
                        hit = float(top in e.targets.split("+"))
                        hits.append(hit)
                    perev_rows.append(dict(detector=D.name, seed=seed, method=mname, event=e.event,
                                           targets=e.targets, single=e.single, top=top, hit=hit,
                                           top_oracle=top_or, hit_oracle=hit_or,
                                           magnitude_z=e.magnitude_z, stealth_tier=str(e.stealth_tier)))
                hits = np.array(hits)
                sing = ev.single.to_numpy()
                lo, hi = boot_ci(hits[sing])
                loc_rows.append(dict(detector=D.name, seed=seed, method=mname,
                                     localized_events=int(np.nansum(hits[sing])),
                                     detected_single=int((~np.isnan(hits[sing])).sum()),
                                     hit_rate_single=float(np.nanmean(hits[sing])),
                                     ci_low=lo, ci_high=hi,
                                     hit_rate_multi=float(np.nanmean(hits[~sing])) if (~np.isnan(hits[~sing])).any() else np.nan))
                for tier in ["high stealth", "medium", "low stealth"]:
                    mk = (ev.stealth_tier == tier).to_numpy()
                    tier_rows.append(dict(detector=D.name, seed=seed, method=mname, tier=tier,
                                          events=int(mk.sum()),
                                          detection_rate=float(detected[mk].mean()),
                                          localization_rate_all=float(np.nan_to_num(hits[mk]).mean()),
                                          localization_rate_detected=float(np.nanmean(hits[mk])) if (~np.isnan(hits[mk])).any() else np.nan))

            # ---- trust gating (residual detectors only; residual attribution is cheap)
            # swept over pre-specified threshold quantiles and noise levels; all reported
            if cal_sd is not None:
                for q in GATE_QS:
                    thr_q = float(np.quantile(cal_s, q)); al_q = s > thr_q
                    aidx = np.flatnonzero(al_q)
                    top0 = np.argmax(attr_residual(det, Xte[aidx], gidx, cal_sd), axis=1)
                    for sd_ in GATE_SDS:
                        rng = np.random.default_rng(seed)
                        same = np.zeros(len(aidx))
                        for _ in range(NOISE_REPS):
                            Xn = Xte[aidx] + rng.normal(0, sd_, size=Xte[aidx].shape).astype(np.float32)
                            same += np.argmax(attr_residual(det, Xn, gidx, cal_sd), axis=1) == top0
                        stable = np.zeros(len(y), bool)
                        stable[aidx] = same >= NOISE_REPS - 1
                        pols = [("trust gate", trust_gate(al_q, stable, files, GATE_K))]
                        if sd_ == GATE_SDS[0]:
                            pols = [("no gate", al_q), ("duration filter", duration_filter(al_q, files, GATE_K))] + pols
                        for policy, al in pols:
                            mm, _ = evaluate(al, y, files, ev)
                            gate_rows.append(dict(detector=D.name, seed=seed, thr_quantile=q,
                                                  noise_sd=sd_ if policy == "trust gate" else np.nan,
                                                  policy=policy,
                                                  stable_share_attack=float(stable[aidx][in_attack[aidx]].mean()),
                                                  stable_share_normal=float(stable[aidx][~in_attack[aidx]].mean()) if (~in_attack[aidx]).any() else np.nan,
                                                  **mm))
                    for k in [1, 5, 10, 20, 30, 60, 90, 120]:
                        mk_, _ = evaluate(duration_filter(al_q, files, k), y, files, ev)
                        curve_rows.append(dict(detector=D.name, seed=seed, thr_quantile=q, k=k,
                                               event_recall=mk_["event_recall"],
                                               false_alarms_per_hour=mk_["false_alarms_per_hour"],
                                               median_delay_s=mk_["median_delay_s"]))
            print(f"[{time.time()-t0:7.0f}s] {D.name} seed {seed}: F1={m['f1']:.3f} "
                  f"event recall={m['event_recall']:.2f} FA/h={m['false_alarms_per_hour']:.2f}", flush=True)

    pd.DataFrame(det_rows).to_csv(f"{a.out}/E1_detection.csv", index=False)
    pd.DataFrame(loc_rows).to_csv(f"{a.out}/E2_localization.csv", index=False)
    pd.DataFrame(perev_rows).to_csv(f"{a.out}/E2_localization_per_event.csv", index=False)
    pd.DataFrame(tier_rows).to_csv(f"{a.out}/E3_stealth_tiers.csv", index=False)
    pd.DataFrame(gate_rows).to_csv(f"{a.out}/E4_trust_gate.csv", index=False)
    pd.DataFrame(curve_rows).to_csv(f"{a.out}/E4_duration_curve.csv", index=False)
    json.dump(dict(runtime_s=time.time() - t0, smooth_s=SMOOTH, grace_s=GRACE, cal_frac=CAL_FRAC,
                   threshold_quantile=THR_Q, gate_k=GATE_K, gate_threshold_quantiles=GATE_QS, gate_noise_sds=GATE_SDS, noise_reps=NOISE_REPS,
                   seeds=SEEDS[:a.seeds]), open(f"{a.out}/run_config.json", "w"), indent=2)


if __name__ == "__main__":
    main()


## 3. Write the analysis script

Aggregates over seeds, runs exact McNemar tests, and saves 300-dpi PNG + TIFF figures.

In [ ]:
%%writefile analyze.py
"""Aggregate pipeline outputs into the paper's tables (mean ± SD over seeds),
paired significance tests, and 300-dpi figures (PNG + TIFF for Frontiers)."""
import json, os, sys
import numpy as np, pandas as pd
from scipy.stats import binomtest
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

R = sys.argv[1] if len(sys.argv) > 1 else "results"
F = f"{R}/figures"; os.makedirs(F, exist_ok=True)
ORDER = ["PCA", "MLP-AE", "IForest", "Z-max", "USAD"]
fmt = lambda m, s: f"{m:.3f} ± {s:.3f}" if not np.isnan(s) and s > 0 else f"{m:.3f}"

def agg(df, keys, cols):
    g = df.groupby(keys, sort=False)
    out = g[cols].mean().add_suffix("_mean").join(g[cols].std().add_suffix("_sd")).reset_index()
    out["n_seeds"] = g.size().values
    return out

# ---- Table 2: detection
d = pd.read_csv(f"{R}/E1_detection.csv")
ORDER = [o for o in ORDER if o in set(d.detector)]
cols = ["precision", "recall", "f1", "auc_pr", "event_recall", "median_delay_s", "false_alarms_per_hour"]
t2 = agg(d, ["detector"], cols); t2.to_csv(f"{R}/T2_detection_summary.csv", index=False)

# ---- Table 3: localization (single-target events), plus oracle
pe = pd.read_csv(f"{R}/E2_localization_per_event.csv")
loc = pd.read_csv(f"{R}/E2_localization.csv")
t3 = agg(loc, ["detector", "method"], ["hit_rate_single", "ci_low", "ci_high", "detected_single", "hit_rate_multi"])
orc = pe[pe.single].groupby(["detector", "method", "seed"]).hit_oracle.mean().reset_index()
orc = orc.groupby(["detector", "method"]).hit_oracle.agg(["mean", "std"]).reset_index().rename(
    columns={"mean": "oracle_hit_mean", "std": "oracle_hit_sd"})
t3 = t3.merge(orc, on=["detector", "method"]); t3.to_csv(f"{R}/T3_localization_summary.csv", index=False)

# ---- paired test: GroupShapley vs Residual / Z-mean (exact McNemar on events;
#      per-event outcome = majority over seeds, detected single-target events only)
tests = []
for det in ORDER:
    sub = pe[(pe.detector == det) & pe.single]
    piv = sub.pivot_table(index="event", columns="method", values="hit", aggfunc="mean")
    for other in ["Residual", "Z-mean"]:
        if other not in piv or "GroupShapley" not in piv:
            continue
        p = piv[["GroupShapley", other]].dropna()
        a = (p.GroupShapley >= 0.5); b = (p[other] >= 0.5)
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        pv = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 > 0 else 1.0
        tests.append(dict(detector=det, comparison=f"GroupShapley vs {other}", events=len(p),
                          gs_only_correct=n01, other_only_correct=n10, exact_mcnemar_p=pv))
# Kernel SHAP (per sensor, summed to processes) vs group Shapley on the same sampled alarm seconds
if os.path.exists(f"{R}/E11_kernelshap_per_event.csv"):
    ks = pd.read_csv(f"{R}/E11_kernelshap_per_event.csv"); ks = ks[ks.single]
    s_ = ks.groupby(["detector", "method", "seed"]).hit.agg(["mean", "count"]).reset_index()
    t3c = s_.groupby(["detector", "method"]).agg(hit_mean=("mean", "mean"), hit_sd=("mean", "std"), events=("count", "mean")).reset_index()
    t3c.to_csv(f"{R}/T3c_kernelshap_summary.csv", index=False)
    for det in ORDER:
        piv = ks[ks.detector == det].pivot_table(index="event", columns="method", values="hit", aggfunc="mean")
        if piv.empty:
            continue
        a = piv["GroupShapley (same seconds)"] >= 0.5; b = piv["KernelSHAP"] >= 0.5
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        pv = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 > 0 else 1.0
        tests.append(dict(detector=det, comparison="GroupShapley vs KernelSHAP (same seconds)", events=len(piv),
                          gs_only_correct=n01, other_only_correct=n10, exact_mcnemar_p=pv))
pd.DataFrame(tests).to_csv(f"{R}/T3b_mcnemar.csv", index=False)

# ---- Table 4: stealth tiers
st = pd.read_csv(f"{R}/E3_stealth_tiers.csv")
t4 = agg(st, ["detector", "method", "tier"], ["detection_rate", "localization_rate_all", "localization_rate_detected"])
t4.to_csv(f"{R}/T4_stealth_summary.csv", index=False)
# oracle localization by tier (explanation independent of detection)
ev = pd.read_csv(f"{R}/events.csv")
ot = pe.groupby(["detector", "method", "seed", "stealth_tier"]).hit_oracle.mean().reset_index()
ot = ot.groupby(["detector", "method", "stealth_tier"]).hit_oracle.agg(["mean", "std"]).reset_index()
ot.to_csv(f"{R}/T4b_oracle_by_tier.csv", index=False)

# ---- Table 5: trust gate
g = pd.read_csv(f"{R}/E4_trust_gate.csv")
g["noise_sd"] = g["noise_sd"].fillna(-1)
t5 = agg(g, ["detector", "thr_quantile", "policy", "noise_sd"],
         ["event_recall", "false_alarms_per_hour", "median_delay_s", "f1", "stable_share_attack", "stable_share_normal"])
t5["noise_sd"] = t5["noise_sd"].replace(-1, np.nan)
t5.to_csv(f"{R}/T5_trust_gate_summary.csv", index=False)

# ---------------------------------------------------------------- figures
# Figures 1-4 exactly as in the paper, 300 dpi, PNG + TIFF (Frontiers: 180 mm wide)
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({"font.size": 9, "font.family": "DejaVu Sans",
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY, LIGHT, DARK = "#2a6fd6", "#bdbcae", "#e3e2dd", "#555555"
W_IN = 7.09  # 180 mm

def save(fig, name):
    fig.savefig(f"{F}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{F}/{name}.tif", dpi=300, bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.close(fig)

audit = json.load(open(f"{R}/dataset_audit.json"))

# ---- Figure 1: study pipeline
fig, ax = plt.subplots(figsize=(W_IN, 3.9)); ax.set_xlim(0, 760); ax.set_ylim(0, 380); ax.axis("off")
def box(cx, y, h, title, lines, main=False):
    ax.add_patch(FancyBboxPatch((cx - 113, 380 - y - h), 226, h, boxstyle="round,pad=0,rounding_size=8",
                                fc="#e6eefb" if main else "white", ec=BLUE if main else DARK, lw=1.6 if main else 0.9))
    ax.text(cx, 380 - y - 20, title, ha="center", va="center", fontsize=8.5, weight="bold")
    for j, l in enumerate(lines):
        ax.text(cx, 380 - y - 38 - 15 * j, l, ha="center", va="center", fontsize=8, color="#333333")
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, 380 - y2), xytext=(x1, 380 - y1), arrowprops=dict(arrowstyle="-|>", color=DARK, lw=0.9))
c1, c2, c3 = 129, 374, 619
box(c1, 30, 70, "Training data", ["HAI 21.03, normal only", f"{audit['fit_rows']:,} rows"])
box(c2, 30, 70, "Four detectors", ["PCA, MLP-AE,", "IForest, Z-max"])
box(c3, 30, 70, "Alarm threshold", ["q-quantile of scores on", f"held-out tail, {audit['cal_rows']:,} rows"])
box(c1, 145, 70, "Test data", [f"{audit['test_rows']:,} rows", f"{audit['events']} real attack events"])
box(c2, 145, 70, "Score and alarm", ["causal 60-s mean,", "alarm above threshold"])
box(c1, 270, 84, "Detection", ["point F1, event recall,", "false alarms per hour", "Table 3"])
box(c2, 270, 84, "Localization (RQ1, RQ2)", ["residual, Z-mean, group", "Shapley vs labels; stealth", "Fig. 2–3, Tables 4–6"], main=True)
box(c3, 270, 84, "Trust gate (RQ3)", ["alarm after 10 stable s", "vs duration filter", "Table 7, Fig. 4"])
arrow(c1 + 113, 65, c2 - 113, 65); arrow(c2 + 113, 65, c3 - 113, 65)
ax.plot([c3, c3, c2 + 120], [380 - 100, 380 - 180, 380 - 180], color=DARK, lw=0.9); arrow(c2 + 120, 180, c2 + 113, 180)
arrow(c1 + 113, 180, c2 - 113, 180)
ax.plot([c2, c2], [380 - 215, 380 - 245], color=DARK, lw=0.9)
ax.plot([c1, c3], [380 - 245, 380 - 245], color=DARK, lw=0.9)
for x in (c1, c2, c3): arrow(x, 245, x, 270)
save(fig, "Fig1_pipeline")

# ---- Figure 2: localization hit rate by detector x method
fig, ax = plt.subplots(figsize=(W_IN, 3.2))
meths = [("Residual", "Residual", LIGHT), ("Z-mean", "Z-mean", GREY), ("GroupShapley", "Group Shapley", BLUE)]; w = 0.27
for j, (m, lab, col) in enumerate(meths):
    for i, det in enumerate(ORDER):
        r = t3[(t3.detector == det) & (t3.method == m)]
        if len(r):
            v = r.hit_rate_single_mean.iloc[0] * 100; x = i + (j - 1) * w
            ax.bar(x, v, w * 0.92, color=col, label=lab if i == 1 else None)
            ax.text(x, v + 1.5, f"{v:.1f}", ha="center", fontsize=7.5)
maj = audit["majority_class_hit_rate"] * 100
ax.axhline(maj, ls="--", c="#888888", lw=0.9, label=f"Always P1 baseline ({maj:.1f}%)")
labels = [f"{d}\n{t3[t3.detector == d].detected_single_mean.iloc[0]:.0f} detected" for d in ORDER]
ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(labels); ax.set_ylim(0, 108); ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.legend(frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
save(fig, "Fig5_localization")

# ---- Figure 3: stealth tiers, MLP-AE (detected, group Shapley, residual)
tiers = ["high stealth", "medium", "low stealth"]; tl = ["High stealth", "Medium", "Low stealth"]
fig, ax = plt.subplots(figsize=(W_IN, 3.2))
sub = t4[t4.detector == "MLP-AE"]
series = [("Detected", lambda t: sub[(sub.method == "GroupShapley") & (sub.tier == t)].detection_rate_mean.iloc[0], GREY),
          ("Localized, group Shapley", lambda t: sub[(sub.method == "GroupShapley") & (sub.tier == t)].localization_rate_detected_mean.iloc[0], BLUE),
          ("Localized, residual", lambda t: sub[(sub.method == "Residual") & (sub.tier == t)].localization_rate_detected_mean.iloc[0], LIGHT)]
n_ev = ev.stealth_tier.value_counts()
for j, (lab, fn, col) in enumerate(series):
    for i, t in enumerate(tiers):
        v = fn(t) * 100; x = i + (j - 1) * 0.27
        ax.bar(x, v, 0.25, color=col, label=lab if i == 0 else None)
        ax.text(x, v + 1.5, f"{v:.1f}", ha="center", fontsize=7.5)
ax.set_xticks(range(3)); ax.set_xticklabels([f"{a}\n{n_ev[t]} events" for a, t in zip(tl, tiers)])
ax.set_ylim(0, 108); ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "Fig6_stealth")

# ---- Figure 4: trust gate vs duration-filter frontier, q = 0.99
c = pd.read_csv(f"{R}/E4_duration_curve.csv")
fig, axs = plt.subplots(1, 2, figsize=(W_IN, 3.0), sharey=True)
best = {"PCA": 0.05, "MLP-AE": 0.5}
for ax, det in zip(axs, ["PCA", "MLP-AE"]):
    cc = c[(c.detector == det) & (c.thr_quantile == 0.99)].groupby("k")[["false_alarms_per_hour", "event_recall"]].mean().sort_values("false_alarms_per_hour")
    ax.plot(cc.false_alarms_per_hour, cc.event_recall * 100, "o-", ms=3, c=GREY, lw=1.2, label="Duration filter, k = 1–120 s")
    for k in (1, 60):
        ax.annotate(f"k = {k} s", (cc.loc[k].false_alarms_per_hour, cc.loc[k].event_recall * 100), textcoords="offset points",
                    xytext=(0, -12 if det == "PCA" else 7), ha="right" if k == 1 else "center", fontsize=7, color="#555555")
    gg = t5[(t5.detector == det) & (t5.thr_quantile == 0.99) & (t5.policy == "trust gate")]
    for _, r in gg.iterrows():
        key = abs(r.noise_sd - best[det]) < 1e-9
        ax.scatter(r.false_alarms_per_hour_mean, r.event_recall_mean * 100, s=40 if key else 18, c=BLUE,
                   alpha=1 if key else 0.45, zorder=3, label="Trust gate" if key else None)
        if key:
            ax.annotate(f"σ = {r.noise_sd:g}: {r.false_alarms_per_hour_mean:.3f}/h, {r.median_delay_s_mean:.1f} s",
                        (r.false_alarms_per_hour_mean, r.event_recall_mean * 100), textcoords="offset points",
                        xytext=(6, 7) if det == "PCA" else (6, -16), ha="left", fontsize=7.5, weight="bold")
    ax.set_title(det, loc="left", fontsize=9, weight="bold"); ax.set_xlabel("False alarms per hour"); ax.set_ylim(28, 103)
    ax.grid(axis="y", color="#eeeeee")
axs[0].set_ylabel("Event recall (%)"); axs[0].legend(frameon=False, fontsize=7, loc="lower right")
save(fig, "Fig7_trust_gate")
print("ok")


## 4. Run the full experiment (5 seeds, about 3–5 minutes)

In [ ]:
!python fdi_pipeline.py --data hai/hai-21.03 --out results --seeds 5 2>&1 | grep -v -i warn
!python analyze.py results

## 4b. Ablations, event map, case studies, cost and architecture (about 1 minute)

Writes Figures 5–8 and Tables 8–9 (E5–E8 CSV files).

In [ ]:
%%writefile extras.py
"""
Additional analyses for the paper (run after fdi_pipeline.py and analyze.py):
  E5  ablation: smoothing window x threshold quantile (PCA)            -> Figure 5, Table 8
  E5b ablation: Shapley reference value, calibration size, occlusion   -> Table 8
  E6  per-event localization map (all detectors/methods, from E2 CSV)  -> Figure 6
  E7  case studies: score, threshold and attribution over time (PCA)   -> Figure 7
  E8  computational cost per detector and attribution                  -> Table 9
  Figure 9: deployment architecture

Usage:
    python extras.py --data hai/hai-21.03 --results results
"""
import argparse, json, os, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fdi_pipeline as fp

plt.rcParams.update({"font.size": 9, "font.family": "DejaVu Sans",
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY, LIGHT, DARK, RED = "#2a6fd6", "#bdbcae", "#e3e2dd", "#555555", "#c0504d"
PROC_COL = {"P1": "#2a6fd6", "P2": "#e8a33d", "P3": "#5aa469", "P4": "#9a8fbf"}
W_IN = 7.09


def save(fig, F, name):
    fig.savefig(f"{F}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{F}/{name}.tif", dpi=300, bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.close(fig)


def attr_occlusion(det, X, gidx, base):
    """Cheaper alternative: score drop when one process is restored to the reference (4 evaluations)."""
    s = det.score(X)
    out = []
    for g in fp.GROUPS:
        Xr = X.copy(); Xr[:, gidx[g]] = base[gidx[g]]
        out.append(s - det.score(Xr))
    return np.stack(out, axis=1)


def event_hits(A_rows, rows_idx, alarm, ev, single_only=True):
    """A_rows: attributions for rows_idx (attack rows). Returns hit array over events (nan = missed)."""
    pos = {r: i for i, r in enumerate(rows_idx)}
    hits = []
    for _, e in ev.iterrows():
        if single_only and not e.single:
            continue
        idx = np.arange(e.start, e.end + 1)
        idx = idx[alarm[idx]]
        if len(idx) == 0:
            hits.append(np.nan); continue
        A = A_rows[[pos[i] for i in idx]].mean(axis=0)
        hits.append(float(fp.GROUPS[int(np.argmax(A))] in e.targets.split("+")))
    return np.array(hits)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--results", default="results")
    a = ap.parse_args()
    R = a.results; F = f"{R}/figures"; os.makedirs(F, exist_ok=True)
    t0 = time.time()

    tr, te = fp.load(a.data)
    feats, gidx, Xfit, Xcal, Xte, base = fp.prepare(tr, te)
    ev = fp.events(te)
    evs = pd.read_csv(f"{R}/events.csv")
    ev["magnitude_z"] = evs.magnitude_z.values; ev["stealth_tier"] = evs.stealth_tier.values
    y = te["attack"].to_numpy().astype(int); files = te["file"].to_numpy()
    att_rows = np.flatnonzero(y == 1)

    # ------------------------------------------------ E5: PCA ablations
    det = fp.PCARecon(42).fit(Xfit)
    raw_te, raw_cal = det.score(Xte), det.score(Xcal)
    cal_sd = np.std(det.resid(Xcal), axis=0) + 1e-6
    Xa = Xte[att_rows]
    refs = {"training median (default)": base, "training mean": np.zeros_like(base),
            "calibration median": np.median(Xcal, axis=0).astype(np.float32)}
    A_gs = {k: fp.attr_group_shapley(det, Xa, gidx, b) for k, b in refs.items()}
    A_res = fp.attr_residual(det, Xa, gidx, cal_sd)
    A_occ = attr_occlusion(det, Xa, gidx, base)
    A_z = fp.attr_z(Xa, gidx)

    grid = []
    for sm in [10, 30, 60, 120]:
        fp.SMOOTH = sm
        s_te = fp.smooth(raw_te, files); s_cal = fp.smooth(raw_cal, np.zeros(len(raw_cal)))
        for q in [0.99, 0.995, 0.999, 0.9995]:
            alarm = s_te > np.quantile(s_cal, q)
            m, _ = fp.evaluate(alarm, y, files, ev)
            h_gs = event_hits(A_gs["training median (default)"], att_rows, alarm, ev)
            h_res = event_hits(A_res, att_rows, alarm, ev)
            grid.append(dict(smooth_s=sm, q=q, event_recall=m["event_recall"], f1=m["f1"],
                             false_alarms_per_hour=m["false_alarms_per_hour"], median_delay_s=m["median_delay_s"],
                             detected_single=int((~np.isnan(h_gs)).sum()),
                             gs_hit=float(np.nanmean(h_gs)), residual_hit=float(np.nanmean(h_res))))
    fp.SMOOTH = 60
    grid = pd.DataFrame(grid); grid.to_csv(f"{R}/E5_ablation_grid.csv", index=False)

    s_te = fp.smooth(raw_te, files); s_cal = fp.smooth(raw_cal, np.zeros(len(raw_cal)))
    alarm = s_te > np.quantile(s_cal, fp.THR_Q)
    rows = []
    for k, A in A_gs.items():
        h = event_hits(A, att_rows, alarm, ev)
        rows.append(dict(factor="Shapley reference value", setting=k, hit=float(np.nanmean(h)), detected=int((~np.isnan(h)).sum())))
    for k, A in [("Group Shapley, 16 evaluations (default)", A_gs["training median (default)"]),
                 ("Occlusion, 4 evaluations", A_occ), ("Residual", A_res), ("Z-mean", A_z)]:
        h = event_hits(A, att_rows, alarm, ev)
        rows.append(dict(factor="Attribution method", setting=k, hit=float(np.nanmean(h)), detected=int((~np.isnan(h)).sum())))
    for cf in [0.1, 0.2, 0.3]:
        fp.CAL_FRAC = cf
        _, gidx2, Xf2, Xc2, Xt2, base2 = fp.prepare(tr, te)
        d2 = fp.PCARecon(42).fit(Xf2)
        al2 = fp.smooth(d2.score(Xt2), files) > np.quantile(fp.smooth(d2.score(Xc2), np.zeros(len(Xc2))), fp.THR_Q)
        m2, _ = fp.evaluate(al2, y, files, ev)
        h2 = event_hits(fp.attr_group_shapley(d2, Xt2[att_rows], gidx2, base2), att_rows, al2, ev)
        rows.append(dict(factor="Calibration tail of last training file", setting=f"{int(cf*100)}%{' (default)' if cf == 0.2 else ''}",
                         hit=float(np.nanmean(h2)), detected=int((~np.isnan(h2)).sum()),
                         event_recall=m2["event_recall"], false_alarms_per_hour=m2["false_alarms_per_hour"]))
    fp.CAL_FRAC = 0.2
    abl = pd.DataFrame(rows); abl.to_csv(f"{R}/E5_ablation_factors.csv", index=False)

    # Figure 5: heatmaps over smoothing x threshold
    fig, axs = plt.subplots(1, 3, figsize=(W_IN, 2.6))
    sms, qs = [10, 30, 60, 120], [0.99, 0.995, 0.999, 0.9995]
    for ax, (col, title, fmt, cmap) in zip(axs, [("event_recall", "Event recall (%)", "{:.0f}", "Greys"),
                                                  ("false_alarms_per_hour", "False alarms per hour", "{:.2f}", "Greys"),
                                                  ("gs_hit", "Group Shapley localization (%)", "{:.0f}", "Blues")]):
        M = grid.pivot(index="smooth_s", columns="q", values=col).loc[sms, qs].to_numpy()
        V = M * 100 if col != "false_alarms_per_hour" else M
        ax.imshow(V, cmap=cmap, aspect="auto", vmin={"event_recall": 60, "gs_hit": 80}.get(col, 0),
                  vmax=(V.max() * 1.3 if col == "false_alarms_per_hour" else 100))
        for i in range(len(sms)):
            for j in range(len(qs)):
                v = V[i, j]; dark = (v > (V.max() * 0.75 if col == "false_alarms_per_hour" else (90 if col == "event_recall" else 92)))
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=7.5, color="white" if dark else "black",
                        weight="bold" if (sms[i] == 60 and qs[j] == 0.999) else "normal")
        ax.set_xticks(range(4)); ax.set_xticklabels([str(q) for q in qs], fontsize=7)
        ax.set_yticks(range(4)); ax.set_yticklabels([f"{s} s" for s in sms], fontsize=7)
        ax.set_title(title, fontsize=8.5, loc="left"); ax.set_xlabel("Threshold quantile q", fontsize=7.5)
        ax.add_patch(plt.Rectangle((qs.index(0.999) - 0.5, sms.index(60) - 0.5), 1, 1, fill=False, ec=RED, lw=1.4))
    axs[0].set_ylabel("Smoothing window")
    save(fig, F, "Fig8_ablation")

    # ------------------------------------------------ E6: per-event map (from pipeline CSV)
    pe = pd.read_csv(f"{R}/E2_localization_per_event.csv")
    combos = [(d, m) for d in ["PCA", "MLP-AE", "IForest", "Z-max"] for m in ["Residual", "Z-mean", "GroupShapley"]
              if not (d == "IForest" and m == "Residual")]
    order = ev.sort_values("magnitude_z").event.tolist()
    S = np.zeros((len(order), len(combos)))
    for j, (d, m) in enumerate(combos):
        sub = pe[(pe.detector == d) & (pe.method == m)]
        for i, e in enumerate(order):
            h = sub[sub.event == e].hit.to_numpy()
            det_frac = np.mean(~np.isnan(h))
            S[i, j] = 0 if det_frac < 0.5 else (2 if np.nanmean(h) >= 0.5 else 1)
    pd.DataFrame(S, index=order, columns=[f"{d}|{m}" for d, m in combos]).to_csv(f"{R}/E6_event_map.csv")
    from matplotlib.colors import ListedColormap
    fig, ax = plt.subplots(figsize=(W_IN, 6.2))
    ax.imshow(S, cmap=ListedColormap(["#f2f2f2", RED, BLUE]), aspect="auto", vmin=0, vmax=2)
    ax.set_xticks(range(len(combos)))
    ax.set_xticklabels([f"{d}\n{'G. Shapley' if m == 'GroupShapley' else m}" for d, m in combos], fontsize=6.5)
    lab = ev.set_index("event").loc[order]
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"{t:<6} {z:6.1f}" for t, z in zip(lab.targets, lab.magnitude_z)], fontsize=5.5, family="monospace")
    for b in [17, 33]:
        ax.axhline(b - 0.5, color=DARK, lw=0.8)
    for j in [2.5, 5.5, 7.5]:
        ax.axvline(j, color="white", lw=2)
    ax.text(len(combos) - 0.4, 8, "high\nstealth", fontsize=7, va="center"); ax.text(len(combos) - 0.4, 25, "medium", fontsize=7, va="center")
    ax.text(len(combos) - 0.4, 41.5, "low\nstealth", fontsize=7, va="center")
    ax.set_ylabel("Attack event: target process and deviation (z), sorted by deviation", fontsize=7.5)
    ax.legend(handles=[Patch(color=BLUE, label="Detected, correct process"), Patch(color=RED, label="Detected, wrong process"),
                       Patch(color="#f2f2f2", label="Not detected")], frameon=False, ncol=3, loc="lower center",
              bbox_to_anchor=(0.5, 1.0), fontsize=7.5)
    save(fig, F, "Fig9_event_map")

    # ------------------------------------------------ E7: case studies (PCA)
    pca_pe = pe[(pe.detector == "PCA")].pivot_table(index="event", columns="method", values="hit").reindex(ev.event.values)
    cand = pca_pe[(pca_pe.GroupShapley == 1) & (pca_pe.Residual == 0)].index.tolist()
    hs = [e for e in ev[(ev.stealth_tier == "high stealth") & ev.single].event if pca_pe.loc[e, "GroupShapley"] == 1] if len(pca_pe) else []
    picks = []
    for e in cand:
        t = ev.loc[e, "targets"]
        if t not in [ev.loc[p, "targets"] for p in picks]:
            picks.append(e)
        if len(picks) == 2:
            break
    if hs:
        picks.append(hs[0])
    picks = picks[:3]
    thr = float(np.quantile(s_cal, fp.THR_Q))
    fig, axs = plt.subplots(3, len(picks), figsize=(W_IN, 5.4), sharex="col", squeeze=False,
                            gridspec_kw={"height_ratios": [1.2, 1, 1]})
    for jj in range(1, len(picks)):
        axs[0, jj].sharey(axs[0, 0]); axs[0, jj].tick_params(labelleft=False)
    case_rows = []
    for j, e in enumerate(picks):
        E = ev.loc[e]; lo = max(int(E.start) - 300, 0); hi = min(int(E.end) + 300, len(y) - 1)
        lo = max(lo, int(np.flatnonzero(files == files[E.start])[0])); hi = min(hi, int(np.flatnonzero(files == files[E.start])[-1]))
        idx = np.arange(lo, hi + 1, 5); t = (idx - E.start)
        ax = axs[0, j]
        ax.plot(t, s_te[idx], color=DARK, lw=0.9); ax.axhline(thr, color=RED, lw=0.8, ls="--")
        ax.axvspan(0, E.end - E.start, color=PROC_COL.get(E.targets.split("+")[0], GREY), alpha=0.15)
        ax.set_yscale("log"); ax.set_title(f"Event {e}: {E.targets} attacked, {E.magnitude_z:.1f} z", fontsize=7.5, loc="left")
        for r, (name, fn) in enumerate([("Group Shapley", lambda X: fp.attr_group_shapley(det, X, gidx, base)),
                                         ("Residual", lambda X: fp.attr_residual(det, X, gidx, cal_sd))]):
            A = np.clip(fn(Xte[idx]), 0, None); share = A / np.maximum(A.sum(axis=1, keepdims=True), 1e-12)
            share = pd.DataFrame(share).rolling(6, min_periods=1).mean().to_numpy()
            ax2 = axs[r + 1, j]
            ax2.stackplot(t, share.T, colors=[PROC_COL[g] for g in fp.GROUPS], alpha=0.9, labels=fp.GROUPS)
            ax2.axvline(0, color="black", lw=0.6); ax2.axvline(E.end - E.start, color="black", lw=0.6)
            ax2.set_ylim(0, 1); ax2.set_yticks([0, 0.5, 1])
            if j == 0:
                ax2.set_ylabel(f"{name}\nattribution share", fontsize=7.5)
            ins = (t >= 0) & (t <= E.end - E.start)
            case_rows.append(dict(event=int(e), targets=E.targets, method=name,
                                  **{f"share_{g}": float(share[ins, k].mean()) for k, g in enumerate(fp.GROUPS)}))
        axs[2, j].set_xlabel("Seconds from attack start", fontsize=7.5)
    axs[0, 0].set_ylabel("PCA anomaly score\n(log, 60-s mean)", fontsize=7.5)
    h_, l_ = axs[1, 0].get_legend_handles_labels()
    fig.legend(h_, [f"{g} " + {"P1": "boiler", "P2": "turbine", "P3": "water treatment", "P4": "HIL"}[g] for g in l_],
               loc="lower center", ncol=4, fontsize=7, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.subplots_adjust(bottom=0.12)
    pd.DataFrame(case_rows).to_csv(f"{R}/E7_case_studies.csv", index=False)
    save(fig, F, "Fig10_case_studies")

    # ------------------------------------------------ E8: computational cost
    cost = []
    Xs = Xte[:20000]; Xa1 = Xa[:1000]
    for D in [fp.PCARecon, fp.MLPAE, fp.IForest, fp.ZMax]:
        t1 = time.perf_counter(); d = D(42).fit(Xfit); fit_t = time.perf_counter() - t1
        t1 = time.perf_counter(); d.score(Xs); sc = (time.perf_counter() - t1) / len(Xs) * 1e3
        rowd = dict(detector=D.name, fit_s=fit_t, score_ms_per_1k=sc * 1000)  # ms per 1,000 samples
        t1 = time.perf_counter(); fp.attr_group_shapley(d, Xa1, gidx, base); rowd["group_shapley_ms_per_1k"] = (time.perf_counter() - t1) * 1000
        t1 = time.perf_counter(); attr_occlusion(d, Xa1, gidx, base); rowd["occlusion_ms_per_1k"] = (time.perf_counter() - t1) * 1000
        if d.resid(Xa1[:2]) is not None:
            sd_ = np.std(d.resid(Xcal[:5000]), axis=0) + 1e-6
            t1 = time.perf_counter(); fp.attr_residual(d, Xa1, gidx, sd_); rowd["residual_ms_per_1k"] = (time.perf_counter() - t1) * 1000
        t1 = time.perf_counter(); fp.attr_z(Xa1, gidx); rowd["zmean_ms_per_1k"] = (time.perf_counter() - t1) * 1000
        cost.append(rowd)
    pd.DataFrame(cost).to_csv(f"{R}/E8_cost.csv", index=False)

    # ------------------------------------------------ Figure 9: deployment architecture
    fig, ax = plt.subplots(figsize=(W_IN, 3.7)); ax.set_xlim(0, 760); ax.set_ylim(0, 370); ax.axis("off")
    def box(x, y, w, h, title, lines, fc="white", ec=DARK, lw=0.9):
        ax.add_patch(FancyBboxPatch((x, 370 - y - h), w, h, boxstyle="round,pad=0,rounding_size=7", fc=fc, ec=ec, lw=lw))
        ax.text(x + w / 2, 370 - y - 16, title, ha="center", va="center", fontsize=7.5, weight="bold")
        for k, l in enumerate(lines):
            ax.text(x + w / 2, 370 - y - 31 - 12 * k, l, ha="center", va="center", fontsize=6.3, color="#333333")
    def arr(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, 370 - y2), xytext=(x1, 370 - y1), arrowprops=dict(arrowstyle="-|>", color=DARK, lw=0.9))
    ax.add_patch(FancyBboxPatch((8, 370 - 26 - 104), 744, 104, boxstyle="round,pad=0,rounding_size=8", fc="#f5f5f2", ec="none"))
    ax.text(16, 370 - 38, "OFFLINE  (normal operation only)", fontsize=6.8, color="#666666", weight="bold")
    W4, G4 = 168, 18
    xs4 = [22 + k * (W4 + G4) for k in range(4)]
    box(xs4[0], 48, W4, 66, "Historian data", ["1-Hz sensor and actuator", "tags (e.g. OPC, Modbus)"])
    box(xs4[1], 48, W4, 66, "Fit detector", ["standardize; any model:", "PCA, AE, IForest, Z-max"])
    box(xs4[2], 48, W4, 66, "Calibrate", ["threshold q on a held-out", "normal tail"])
    box(xs4[3], 48, W4, 66, "Process map", ["sensor -> process group", "(P1 ... P4)"])
    for k in range(3):
        arr(xs4[k] + W4, 81, xs4[k + 1], 81)
    ax.text(16, 370 - 152, "ONLINE  (every second)", fontsize=6.8, color="#666666", weight="bold")
    W5, G5 = 132, 15
    xs5 = [22 + k * (W5 + G5) for k in range(5)]
    box(xs5[0], 165, W5, 70, "Live tags", ["buffered and", "standardized"])
    box(xs5[1], 165, W5, 70, "Detector score", ["causal 60-s mean,", "compared with q"])
    box(xs5[2], 165, W5, 70, "Group Shapley", ["16 evaluations;", "top = location"], fc="#e6eefb", ec=BLUE, lw=1.5)
    box(xs5[3], 165, W5, 70, "Trust gate", ["alert after 10", "stable seconds"], fc="#e6eefb", ec=BLUE, lw=1.5)
    box(xs5[4], 165, W5, 70, "Operator alert", ["process, attribution", "share, stability"])
    for k in range(4):
        arr(xs5[k] + W5, 200, xs5[k + 1], 200)
    ax.text(xs5[1] + W5 / 2, 370 - 250, "below threshold: no alert", fontsize=6.2, color="#666666", ha="center")
    ax.text(xs5[3] + W5 / 2, 370 - 250, "unstable: hold and re-check", fontsize=6.2, color="#666666", ha="center")
    arr(xs4[1] + W4 / 2, 114, xs5[1] + W5 / 2, 165); arr(xs4[2] + W4 / 2, 114, xs5[1] + W5 * 0.8, 165)
    arr(xs4[3] + W4 / 2, 114, xs5[2] + W5 / 2, 165)
    box(190, 278, 380, 62, "Response (outside this study)", ["inspect or isolate the named process;", "log the alert, attribution and stability"])
    arr(xs5[4] + W5 / 2, 235, 570, 300)
    save(fig, F, "Fig12_architecture")

    json.dump(dict(runtime_s=time.time() - t0, case_events=[int(p) for p in picks]), open(f"{R}/extras_config.json", "w"), indent=2)
    print("extras ok", round(time.time() - t0), "s; case events", picks)


if __name__ == "__main__":
    main()


In [ ]:
!python extras.py --data hai/hai-21.03 --results results 2>&1 | grep -v -i warn

## 5. Show the figures

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f)
    display(Image(f, width=900))

## 6. Show the result tables used in the paper

In [ ]:
import pandas as pd, json
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(json.dumps(json.load(open('results/dataset_audit.json')), indent=2))
cols = {
  'T2_detection_summary': ['detector','precision_mean','recall_mean','f1_mean','f1_sd','auc_pr_mean','event_recall_mean','event_recall_sd','median_delay_s_mean','false_alarms_per_hour_mean'],
  'T3_localization_summary': ['detector','method','hit_rate_single_mean','hit_rate_single_sd','ci_low_mean','ci_high_mean','detected_single_mean','hit_rate_multi_mean','oracle_hit_mean'],
  'T3b_mcnemar': None,
  'T4_stealth_summary': ['detector','method','tier','detection_rate_mean','localization_rate_detected_mean'],
  'T5_trust_gate_summary': ['detector','thr_quantile','policy','noise_sd','event_recall_mean','false_alarms_per_hour_mean','median_delay_s_mean','stable_share_attack_mean','stable_share_normal_mean'],
}
for t, c in cols.items():
    d = pd.read_csv(f'results/{t}.csv')
    print('\n==', t)
    display((d[c] if c else d).round(3))
for t in ['E5_ablation_factors','E5_ablation_grid','E8_cost']:
    print('\n==', t); display(pd.read_csv(f'results/{t}.csv').round(3))


## 7. Download all results (CSV, JSON, figures) as a zip

In [ ]:
!zip -qr results.zip results
try:
    from google.colab import files
    files.download('results.zip')
except ImportError:
    print('results.zip created (download is only automatic in Colab)')

## 8. Second dataset: HAI 22.04 (process, control-loop and data-point localization)

Downloads HAI 22.04 (Git LFS, ~800 MB), writes the ground-truth table built from the HAI technical details document, and runs all four detectors with five seeds (about 12–15 minutes).

In [ ]:
!apt-get -qq install git-lfs > /dev/null 2>&1; git lfs install > /dev/null
!cd hai && git lfs pull --include="hai-22.04/*.csv" && git checkout HEAD -- hai-22.04 hai_dataset_technical_details.pdf
!ls -lh hai/hai-22.04/*.csv

In [ ]:
%%writefile hai22_ground_truth.csv
id,file,order,scenarios,loops,points,ambiguous,note
A101,test1,1,AP04,P1-PC,P1_PCV01D,0,
A102,test1,2,AP18,P1-LC,P1_LCV01D,0,
A103,test1,3,AP11,P1-FC,P1_FCV03D;P1_FT03,0,
A104,test1,4,AP37,P3-LC,P3_LCV01D,0,
A105,test1,5,AP14,P1-LC,P1_B3004,0,
A106,test1,6,AP13,P1-FC;P1-CC,P1_FCV03D;P1_PP04,1,PDF lists scenario AP13 (P1-FC) but target P1-CC/P1_PP04
A107,test1,7,AP19,P1-TC,P1_FCV01D,0,
A201,test2,1,AP01,P1-PC,P1_B2016,0,
A202,test2,2,AP13,P1-FC,P1_FCV03D,0,
A203,test2,3,AP31,P2-SC,P2_AutoSD,0,
A204,test2,4,AP04;AP29,P1-PC;P2-SC,P1_PCV01D;P2_SCO,0,
A205,test2,5,AP37,P3-LC,P3_LCV01D,0,
A206,test2,6,AP02;AP27,P1-PC;P2-SC,P1_B2016;P1_PIT01;P2_AutoSD;P2_SIT01,0,
A207,test2,7,AP16,P1-LC,P1_LCV01D,0,
A208,test2,8,AP30,P2-SC,P2_SCO;P2_SIT01,0,
A209,test2,9,AP03,P1-PC,P1_B2016;P1_PIT01,0,PDF also lists P1_FIT01 which is not a data point
A210,test2,10,AP26,P2-SC,P2_AutoSD,0,
A211,test2,11,AP05,P1-PC,P1_PCV01D;P1_PIT01,0,
A212,test2,12,AP35,P3-LC,P3_LCP01D,0,
A213,test2,13,AP24,P1-CC,P1_PP04,0,
A214,test2,14,AP39,P3-LC,P3_LCV01D,0,
A215,test2,15,AP09,P1-FC,P1_B3005;P1_FT03,0,
A216,test2,16,AP01;AP08,P1-PC;P1-FC,P1_B2016;P1_B3005,0,
A217,test2,17,AP10,P1-FC,P1_FCV03D,0,
A301,test3,1,AP16;AP10,P1-LC;P3-LC;P1-FC,P1_LCV01D;P3_LCV01D;P1_FCV03D,1,PDF lists AP16 (P1-LC) with controller P3-LC and point P2_LCV01D
A302,test3,2,AP15,P1-LC,P1_B3004;P1_LIT01,0,points taken from scenario AP15 (PDF row shifted)
A303,test3,3,AP17;AP37,P1-LC;P3-LC,P1_LCV01D;P1_LIT01;P3_LCV01D,0,points taken from scenarios (PDF row shifted)
A304,test3,4,AP38,P3-LC,P3_LCV01D;P3_LIT01,0,points taken from scenario AP38 (PDF row shifted)
A305,test3,5,AP18,P1-LC,P1_LCV01D,0,points taken from scenario AP18 (PDF row shifted)
A306,test3,6,AP04;AP15,P1-PC;P1-LC,P1_PCV01D;P1_B3004;P1_LIT01,0,points taken from scenarios (PDF row shifted)
A307,test3,7,AP20,P1-TC,P1_FCV01D;P1_TIT01,0,
A308,test3,8,AP05;AP23,P1-PC;P1-CC,P1_PCV01D;P1_PIT01;P1_PP04,0,
A309,test3,9,AP08;AP19,P1-FC;P1-TC,P1_B3005;P1_FCV01D,0,
A310,test3,10,AP35;AP37,P3-LC,P3_LCP01D;P3_LCV01D,0,
A401,test4,1,AP28,P2-SC,P2_ManualSD,0,
A402,test4,2,AP21,P1-TC,P1_FCV01D,0,
A403,test4,3,AP34,P2-TR,P2_RTR,0,
A404,test4,4,AP26;AP37,P2-SC;P3-LC,P2_AutoSD;P3_LCV01D,0,
A405,test4,5,AP22,P1-TC,P1_B4002,0,
A406,test4,6,AP09;AP19,P1-FC;P1-TC,P1_B3005;P1_FT03;P1_FCV01D,0,
A407,test4,7,AP13,P1-FC,P1_FCV03D,0,
A408,test4,8,AP17;AP05,P1-LC;P1-PC,P1_LCV01D;P1_LIT01;P1_PCV01D;P1_PIT01,0,
A409,test4,9,AP18;AP21,P1-LC;P1-TC,P1_LCV01D;P1_FCV01D,0,
A410,test4,10,AP11;AP27,P1-FC;P2-SC,P1_FCV03D;P1_FT03;P2_AutoSD;P2_SIT01,0,
A411,test4,11,AP23;AP34,P1-CC;P2-TR,P1_PP04;P2_RTR,0,
A412,test4,12,AP20;AP01,P1-TC;P1-PC,P1_FCV01D;P1_TIT01;P1_B2016,0,
A413,test4,13,AP16;AP27,P1-LC;P2-SC,P1_LCV01D;P2_AutoSD;P2_SIT01,0,
A414,test4,14,AP33;AP36,P2-TR;P3-LC,P2_VTR02;P3_LCP01D;P3_LIT01,0,
A415,test4,15,AP33,P2-TR,P2_VTR02,0,PDF lists scenario as AP3 (AP33)
A416,test4,16,AP12,P1-FC,P1_FCV03D;P1_FT03;P1_LIT01,0,
A417,test4,17,AP25,P1-CC,P1_PP04SP,0,
A418,test4,18,AP01;AP14,P1-PC;P1-LC,P1_B2016;P1_B3004,0,
A419,test4,19,AP01;AP35,P1-PC;P3-LC,P1_B2016;P3_LCP01D,0,
A420,test4,20,AP07,P1-PC,P1_PCV01D,0,
A421,test4,21,AP30;AP23,P2-SC;P1-CC,P2_SCO;P2_SIT01;P1_PP04,0,
A422,test4,22,AP02;AP26,P1-PC;P2-SC,P1_B2016;P1_PIT01;P2_AutoSD,0,
A423,test4,23,AP08;AP29,P1-FC;P2-SC,P1_B3005;P2_SCO,0,
A424,test4,24,AP10;AP23,P1-FC;P1-CC,P1_FCV03D;P1_PP04,0,


In [ ]:
%%writefile hai22_pipeline.py
"""
HAI 22.04: detection and multi-level localization (process -> control loop -> data point)
against the attack schedule of the HAI technical details document (hai22_ground_truth.csv).

Usage:
    python hai22_pipeline.py --data /path/to/hai-22.04 --gt hai22_ground_truth.csv --out results22
"""
import argparse, glob, json, os, sys, time
from math import factorial
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fdi_pipeline as fp          # detectors, smoothing, evaluation, bootstrap (same code as HAI 21.03)

SEEDS = [42, 123, 2026, 7, 99]
STRIDE = 3                          # attribute every 3rd attack second (attributions are averaged per event)
PROCS = ["P1", "P2", "P3", "P4"]
# control loops documented in the HAI technical details (boiler, turbine, water treatment); rest grouped
LOOPS = {
    "P1-PC": ["P1_B2016", "P1_PIT01", "P1_PIT01_HH", "P1_PIT02", "P1_PCV01D", "P1_PCV01Z", "P1_PCV02D", "P1_PCV02Z"],
    "P1-LC": ["P1_B3004", "P1_LIT01", "P1_LCV01D", "P1_LCV01Z"],
    "P1-FC": ["P1_B3005", "P1_FT03", "P1_FT03Z", "P1_FCV03D", "P1_FCV03Z"],
    "P1-TC": ["P1_B4022", "P1_B4002", "P1_TIT01", "P1_FCV01D", "P1_FCV01Z", "P1_FCV02D", "P1_FCV02Z", "P1_FT02", "P1_FT02Z"],
    "P1-CC": ["P1_PP04", "P1_PP04SP", "P1_TIT03"],
    "P2-SC": ["P2_AutoSD", "P2_ManualSD", "P2_SCO", "P2_SIT01", "P2_SCST", "P2_AutoGO", "P2_ManualGO"],
    "P2-TR": ["P2_RTR", "P2_VTR01", "P2_VTR02", "P2_VTR03", "P2_VTR04", "P2_VIBTR01", "P2_VIBTR02",
              "P2_VIBTR03", "P2_VIBTR04", "P2_Emerg", "P2_TripEx", "P2_VT01"],
}
LOOP_ORDER = ["P1-PC", "P1-LC", "P1-FC", "P1-TC", "P1-CC", "P1-other", "P2-SC", "P2-TR", "P2-other", "P3-LC", "P4-HIL"]


def load(d):
    tr = [pd.read_csv(f) for f in sorted(glob.glob(f"{d}/train*.csv"))]
    te = []
    for i, f in enumerate(sorted(glob.glob(f"{d}/test*.csv"))):
        x = pd.read_csv(f); x["file"] = i; x["fname"] = os.path.basename(f)[:-4]; te.append(x)
    return tr, pd.concat(te, ignore_index=True)


def prepare(tr, te, cal_frac=0.2):
    sens = [c for c in tr[0].columns if c[:2] in PROCS]
    trall = pd.concat(tr, ignore_index=True)
    feats = [c for c in sens if trall[c].nunique(dropna=True) > 1]
    n = len(tr[-1]); cut = int(n * (1 - cal_frac))
    fit_df = pd.concat(tr[:-1] + [tr[-1].iloc[:cut]], ignore_index=True); cal_df = tr[-1].iloc[cut:]
    mu, sig = fit_df[feats].mean(), fit_df[feats].std(); sig = sig.where(sig > 1e-8, 1.0)
    z = lambda d: ((d[feats] - mu) / sig).to_numpy(np.float32)
    Xfit = z(fit_df)
    return feats, Xfit, z(cal_df), z(te), np.median(Xfit, axis=0).astype(np.float32), len(trall), len(fit_df), len(cal_df)


def groups_for(feats):
    pidx = {p: np.array([i for i, c in enumerate(feats) if c.startswith(p)]) for p in PROCS}
    loop_of = {}
    for L, cols in LOOPS.items():
        for c in cols:
            loop_of[c] = L
    lidx = {}
    for L in LOOP_ORDER:
        if L == "P1-other":
            m = [i for i, c in enumerate(feats) if c.startswith("P1") and c not in loop_of]
        elif L == "P2-other":
            m = [i for i, c in enumerate(feats) if c.startswith("P2") and c not in loop_of]
        elif L == "P3-LC":
            m = [i for i, c in enumerate(feats) if c.startswith("P3")]
        elif L == "P4-HIL":
            m = [i for i, c in enumerate(feats) if c.startswith("P4")]
        else:
            m = [i for i, c in enumerate(feats) if loop_of.get(c) == L]
        if m:
            lidx[L] = np.array(m)
    return pidx, lidx


def events(te, gt):
    a = te["Attack"].to_numpy()
    out = []
    for i, f in enumerate(sorted(te.fname.unique())):
        idx = np.flatnonzero(te.fname.to_numpy() == f)
        aa = a[idx]
        s = np.flatnonzero(np.diff(np.r_[0, aa]) == 1); e = np.flatnonzero(np.diff(np.r_[aa, 0]) == -1)
        g = gt[gt.file == f].sort_values("order")
        assert len(g) == len(s), f"{f}: {len(s)} events in CSV vs {len(g)} in schedule"
        for (si, ei), (_, r) in zip(zip(s, e), g.iterrows()):
            loops = r.loops.split(";")
            out.append(dict(id=r.id, start=int(idx[si]), end=int(idx[ei]), dur=int(ei - si + 1),
                            start_time=te.timestamp.iloc[idx[si]], loops=r.loops, points=r.points,
                            procs=";".join(sorted({l[:2] for l in loops})), ambiguous=int(r.ambiguous),
                            single_loop=int(len(loops) == 1), single_proc=int(len({l[:2] for l in loops}) == 1)))
    return pd.DataFrame(out)


def shapley(det, X, groups, base):
    """Exact Shapley values of det.score over a list of feature groups (2^G coalitions)."""
    G, n = len(groups), len(X)
    v = np.empty((2 ** G, n))
    for mask in range(2 ** G):
        Xs = np.repeat(base[None, :], n, axis=0)
        for g in range(G):
            if mask >> g & 1:
                Xs[:, groups[g]] = X[:, groups[g]]
        v[mask] = det.score(Xs)
    w = [factorial(k) * factorial(G - k - 1) / factorial(G) for k in range(G)]
    pc = np.array([bin(m).count("1") for m in range(2 ** G)])
    phi = np.zeros((n, G))
    for g in range(G):
        for mask in range(2 ** G):
            if not mask >> g & 1:
                phi[:, g] += w[pc[mask]] * (v[mask | (1 << g)] - v[mask])
    return phi


def occlusion(det, X, groups, base):
    s = det.score(X); out = []
    for gi in groups:
        Xr = X.copy(); Xr[:, gi] = base[gi]; out.append(s - det.score(Xr))
    return np.stack(out, axis=1)


def group_mean(M, groups):
    return np.stack([M[:, gi].mean(axis=1) for gi in groups], axis=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True); ap.add_argument("--gt", required=True)
    ap.add_argument("--out", default="results22"); ap.add_argument("--seeds", type=int, default=5)
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True); t0 = time.time()

    gt = pd.read_csv(a.gt)
    tr, te = load(a.data)
    feats, Xfit, Xcal, Xte, base, n_train, n_fit, n_cal = prepare(tr, te)
    pidx, lidx = groups_for(feats)
    P_names, L_names = list(pidx), list(lidx)
    Pg, Lg = [pidx[p] for p in P_names], [lidx[L] for L in L_names]
    ev = events(te, gt)
    y = te["Attack"].to_numpy().astype(int); files = te["file"].to_numpy()

    # stealth magnitude: 95th pct of max |z| over the target loops' sensors
    mag = []
    for _, e in ev.iterrows():
        cols = np.concatenate([lidx[L] for L in e.loops.split(";") if L in lidx])
        mag.append(float(np.percentile(np.abs(Xte[e.start:e.end + 1][:, cols]).max(axis=1), 95)))
    ev["magnitude_z"] = mag
    ev["stealth_tier"] = pd.qcut(ev.magnitude_z, 3, labels=["high stealth", "medium", "low stealth"]).astype(str)
    ev.to_csv(f"{a.out}/events22.csv", index=False)

    valid_pts = lambda e: [feats.index(p) for p in e.points.split(";") if p in feats]
    sl = ev[(ev.single_loop == 1) & (ev.ambiguous == 0)]
    sp = ev[(ev.single_proc == 1)]
    audit = dict(features_used=len(feats), dropped_constant=86 - len(feats), train_rows=n_train, fit_rows=n_fit,
                 cal_rows=n_cal, test_rows=len(Xte), attack_rows=int(y.sum()), events=len(ev),
                 ambiguous=int(ev.ambiguous.sum()), single_loop_events=len(sl), single_process_events=len(sp),
                 loop_groups={L: int(len(lidx[L])) for L in L_names}, process_groups={p: int(len(pidx[p])) for p in P_names},
                 majority_loop=sl.loops.value_counts().index[0], majority_loop_rate=float(sl.loops.value_counts().iloc[0] / len(sl)),
                 majority_process=sp.procs.value_counts().index[0], majority_process_rate=float(sp.procs.value_counts().iloc[0] / len(sp)),
                 target_points_missing=sorted({p for e in ev.itertuples() for p in e.points.split(";") if p not in feats}))
    json.dump(audit, open(f"{a.out}/audit22.json", "w"), indent=2)
    print(json.dumps(audit, indent=1), flush=True)

    # attack seconds used for attribution (every STRIDE-th second of each event)
    rows = np.concatenate([np.arange(e.start, e.end + 1)[::STRIDE] for _, e in ev.iterrows()])
    Xa = Xte[rows]

    det_rows, per_ev = [], []
    for D in [fp.PCARecon, fp.MLPAE, fp.IForest, fp.ZMax]:
        seeds = SEEDS[:a.seeds] if D in (fp.MLPAE, fp.IForest) else SEEDS[:1]
        for seed in seeds:
            det = D(seed).fit(Xfit)
            s_cal = fp.smooth(det.score(Xcal), np.zeros(len(Xcal)))
            s = fp.smooth(det.score(Xte), files); alarm = s > np.quantile(s_cal, fp.THR_Q)
            evq = ev.rename(columns={})
            m, detected = fp.evaluate(alarm, y, files, ev.assign(targets=""), s)
            det_rows.append(dict(detector=D.name, seed=seed, **m))

            r = det.resid(Xcal); cal_sd = (np.std(r, axis=0) + 1e-6) if r is not None else None
            Rres = (det.resid(Xa) / cal_sd) ** 2 if cal_sd is not None else None
            Z = np.abs(Xa)
            A = {("process", "GroupShapley"): shapley(det, Xa, Pg, base),
                 ("process", "Z-mean"): group_mean(Z, Pg),
                 ("loop", "GroupShapley"): shapley(det, Xa, Lg, base),
                 ("loop", "Occlusion"): occlusion(det, Xa, Lg, base),
                 ("loop", "Z-mean"): group_mean(Z, Lg)}
            if Rres is not None:
                A[("process", "Residual")] = group_mean(Rres, Pg); A[("loop", "Residual")] = group_mean(Rres, Lg)
            for k, e in enumerate(ev.itertuples()):
                pos = np.flatnonzero((rows >= e.start) & (rows <= e.end))
                al = pos[alarm[rows[pos]]]
                for (lvl, meth), M in A.items():
                    names = P_names if lvl == "process" else L_names
                    targets = e.procs.split(";") if lvl == "process" else e.loops.split(";")
                    top_or = names[int(np.argmax(M[pos].mean(axis=0)))]
                    top = names[int(np.argmax(M[al].mean(axis=0)))] if len(al) else None
                    per_ev.append(dict(detector=D.name, seed=seed, level=lvl, method=meth, id=e.id,
                                       detected=int(len(al) > 0), top=top, hit=(float(top in targets) if top else np.nan),
                                       top_oracle=top_or, hit_oracle=float(top_or in targets)))
                # data-point level: does a target point appear in the top-k ranked points?
                vp = valid_pts(e)
                if vp and not e.ambiguous:
                    rank_src = {"Z-score": Z}
                    if Rres is not None:
                        rank_src["Residual"] = Rres
                    for meth, M in rank_src.items():
                        for src, P in [("alarm", al), ("oracle", pos)]:
                            if len(P) == 0:
                                continue
                            order = np.argsort(-M[P].mean(axis=0))
                            per_ev.append(dict(detector=D.name, seed=seed, level=f"point-{src}", method=meth, id=e.id,
                                               detected=int(len(al) > 0), top=feats[order[0]],
                                               hit=float(order[0] in vp), hit3=float(bool(set(order[:3]) & set(vp))),
                                               hit5=float(bool(set(order[:5]) & set(vp)))))
                    # hierarchical: loop by group Shapley, then point by z-score inside that loop
                    for src, P in [("alarm", al), ("oracle", pos)]:
                        if len(P) == 0:
                            continue
                        Lt = L_names[int(np.argmax(A[("loop", "GroupShapley")][P].mean(axis=0)))]
                        inside = lidx[Lt]; zz = Z[P][:, inside].mean(axis=0); order = inside[np.argsort(-zz)]
                        per_ev.append(dict(detector=D.name, seed=seed, level=f"point-{src}", method="Loop Shapley + Z-score",
                                           id=e.id, detected=int(len(al) > 0), top=feats[order[0]], hit=float(order[0] in vp),
                                           hit3=float(bool(set(order[:3]) & set(vp))), hit5=float(bool(set(order[:5]) & set(vp)))))
            print(f"[{time.time()-t0:6.0f}s] {D.name} seed {seed}: F1={m['f1']:.3f} event recall={m['event_recall']:.2f} "
                  f"FA/h={m['false_alarms_per_hour']:.3f}", flush=True)
    pd.DataFrame(det_rows).to_csv(f"{a.out}/H1_detection.csv", index=False)
    pd.DataFrame(per_ev).to_csv(f"{a.out}/H2_localization_per_event.csv", index=False)
    json.dump(dict(runtime_s=time.time() - t0, stride=STRIDE, seeds=SEEDS[:a.seeds], loops=L_names), open(f"{a.out}/run22.json", "w"), indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze22.py
"""Summaries, tests and figures for HAI 22.04 (run after hai22_pipeline.py)."""
import json, os, sys
import numpy as np, pandas as pd
from scipy.stats import binomtest
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

R = sys.argv[1] if len(sys.argv) > 1 else "results22"
F = f"{R}/figures"; os.makedirs(F, exist_ok=True)
plt.rcParams.update({"font.size": 9, "font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY, LIGHT = "#2a6fd6", "#bdbcae", "#e3e2dd"
DETS = ["PCA", "MLP-AE", "IForest", "Z-max", "USAD"]

audit = json.load(open(f"{R}/audit22.json"))
ev = pd.read_csv(f"{R}/events22.csv")
d = pd.read_csv(f"{R}/H1_detection.csv")
DETS = [x for x in DETS if x in set(d.detector)]
g = d.groupby("detector", sort=False)
cols = ["precision", "recall", "f1", "auc_pr", "event_recall", "median_delay_s", "false_alarms_per_hour"]
t10 = g[cols].mean().add_suffix("_mean").join(g[cols].std().add_suffix("_sd")).reset_index()
t10.to_csv(f"{R}/T10_detection22.csv", index=False)

pe = pd.read_csv(f"{R}/H2_localization_per_event.csv").merge(
    ev[["id", "single_loop", "single_proc", "ambiguous", "stealth_tier"]], on="id")
rows = []
for lvl, col in [("process", "single_proc"), ("loop", "single_loop")]:
    sub = pe[(pe.level == lvl) & (pe[col] == 1) & (pe.ambiguous == 0)]
    per_seed = sub.groupby(["detector", "method", "seed"]).agg(hit=("hit", "mean"), oracle=("hit_oracle", "mean"),
                                                               detected=("detected", "sum")).reset_index()
    s = per_seed.groupby(["detector", "method"]).agg(hit_mean=("hit", "mean"), hit_sd=("hit", "std"),
                                                     oracle_mean=("oracle", "mean"), oracle_sd=("oracle", "std"),
                                                     detected=("detected", "mean")).reset_index()
    s["level"] = lvl; rows.append(s)
t11 = pd.concat(rows); t11.to_csv(f"{R}/T11_localization22.csv", index=False)

pt = pe[pe.level == "point-oracle"].groupby(["detector", "method", "seed"])[["hit", "hit3", "hit5"]].mean().reset_index()
t12 = pt.groupby(["detector", "method"])[["hit", "hit3", "hit5"]].mean().reset_index()
t12["events"] = pe[pe.level == "point-oracle"].groupby(["detector", "method", "seed"]).id.count().groupby(["detector", "method"]).mean().values
t12.to_csv(f"{R}/T12_points22.csv", index=False)

# exact McNemar on oracle hits (every attack scored), majority over seeds
tests = []
for lvl, col in [("process", "single_proc"), ("loop", "single_loop")]:
    for det in DETS:
        sub = pe[(pe.level == lvl) & (pe[col] == 1) & (pe.ambiguous == 0) & (pe.detector == det)]
        piv = sub.pivot_table(index="id", columns="method", values="hit_oracle", aggfunc="mean")
        for other in ["Residual", "Z-mean", "Occlusion"]:
            if other in piv and "GroupShapley" in piv:
                a_ = piv.GroupShapley >= 0.5; b_ = piv[other] >= 0.5
                n01, n10 = int((a_ & ~b_).sum()), int((~a_ & b_).sum())
                p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
                tests.append(dict(level=lvl, detector=det, comparison=f"GroupShapley vs {other}", events=len(piv),
                                  gs_only=n01, other_only=n10, p=p))
if os.path.exists(f"{R}/H11_kernelshap_per_event.csv"):
    ks = pd.read_csv(f"{R}/H11_kernelshap_per_event.csv").merge(ev[["id", "single_loop", "single_proc", "ambiguous"]], on="id")
    rows_ = []
    for lvl, col in [("process", "single_proc"), ("loop", "single_loop")]:
        sub = ks[(ks.level == lvl) & (ks[col] == 1) & (ks.ambiguous == 0)]
        ps = sub.groupby(["detector", "method", "seed"]).hit_oracle.mean().reset_index()
        r_ = ps.groupby(["detector", "method"]).hit_oracle.agg(["mean", "std"]).reset_index(); r_["level"] = lvl; rows_.append(r_)
        for det in DETS:
            piv = sub[sub.detector == det].pivot_table(index="id", columns="method", values="hit_oracle", aggfunc="mean")
            if piv.empty:
                continue
            a_ = piv["GroupShapley (same seconds)"] >= 0.5; b_ = piv["KernelSHAP"] >= 0.5
            n01, n10 = int((a_ & ~b_).sum()), int((~a_ & b_).sum())
            p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
            tests.append(dict(level=lvl, detector=det, comparison="GroupShapley vs KernelSHAP (same seconds)", events=len(piv),
                              gs_only=n01, other_only=n10, p=p))
    pd.concat(rows_).to_csv(f"{R}/T11b_kernelshap22.csv", index=False)
pd.DataFrame(tests).to_csv(f"{R}/T13_mcnemar22.csv", index=False)

# Figure 8: localization at process and loop level, oracle (all attack seconds), with majority baselines
fig, axs = plt.subplots(1, 2, figsize=(7.09, 3.4), sharey=True)
meths = [("Residual", LIGHT), ("Z-mean", GREY), ("Occlusion", "#8fb3e8"), ("GroupShapley", BLUE)]
for ax, lvl, base_, lab in [(axs[0], "process", audit["majority_process_rate"], "Process (4 groups)"),
                            (axs[1], "loop", audit["majority_loop_rate"], "Control loop (11 groups)")]:
    sub = t11[t11.level == lvl]; ms = [m for m in meths if m[0] in sub.method.unique()]; w = 0.8 / len(ms)
    for j, (m, c) in enumerate(ms):
        for i, det in enumerate(DETS):
            r = sub[(sub.detector == det) & (sub.method == m)]
            if len(r):
                v = r.oracle_mean.iloc[0] * 100; x = i + (j - (len(ms) - 1) / 2) * w
                ax.bar(x, v, w * 0.92, color=c, label=("Group Shapley" if m == "GroupShapley" else m) if i == 1 else None, zorder=3)
    ax.axhline(base_ * 100, ls="--", c="#555555", lw=1.0, zorder=4, label=f"Majority baseline ({base_*100:.1f}%)")
    ax.set_xticks(range(len(DETS))); ax.set_xticklabels(DETS, fontsize=8); ax.set_title(lab, loc="left", fontsize=9, weight="bold")
    ax.set_ylim(0, 100); ax.set_yticks([0, 20, 40, 60, 80, 100]); ax.grid(axis="y", color="#eeeeee", zorder=0)
    ax.legend(frameon=False, fontsize=6.5, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.07))
axs[0].set_ylabel("Correct target, % of single-target attacks\n(every attack second explained)", fontsize=7.5)
fig.savefig(f"{F}/Fig11_hai22_localization.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{F}/Fig11_hai22_localization.tif", dpi=300, bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
print("ok")


In [ ]:
!python hai22_pipeline.py --data hai/hai-22.04 --gt hai22_ground_truth.csv --out results22 --seeds 5 2>&1 | grep -v -i warn | tail -15
!python analyze22.py results22

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image('results22/figures/Fig11_hai22_localization.png', width=900))
for t in ['T10_detection22','T11_localization22','T12_points22','T13_mcnemar22']:
    print('\n==', t); display(pd.read_csv(f'results22/{t}.csv').round(3))
!zip -qr results22.zip results22
try:
    from google.colab import files; files.download('results22.zip')
except ImportError:
    print('results22.zip created')

## 9. Colour diagrams (Figures 1–4 and 12)

Run after sections 4, 4b and 8. Numbers in the diagrams are read from the results folders; this replaces Fig1 and Fig12 with colour versions and adds Fig2–Fig4.

In [ ]:
%%writefile diagrams.py
"""Colour diagrams for the paper (run after fdi_pipeline.py, analyze.py, extras.py and, for Figure 2, hai22_pipeline.py).
  Figure 1  study pipeline
  Figure 2  HAI grouping hierarchy: process -> control loop -> data point
  Figure 3  exact group Shapley over four processes (with a real case-study attribution)
  Figure 4  trust gate versus duration filter
  Figure 12 deployment architecture
Numbers shown in the diagrams are read from the pipeline outputs.

Usage:
    python diagrams.py --results results --results22 results22
"""
import argparse, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

plt.rcParams.update({"font.size": 9, "font.family": "DejaVu Sans"})
W_IN = 7.09  # 180 mm
INK, MUTED = "#1f2937", "#4b5563"
PROC = {"P1": "#2a6fd6", "P2": "#e8902a", "P3": "#3f9c55", "P4": "#8b6fc4"}
PROC_LIGHT = {"P1": "#dbe7fb", "P2": "#fcebd6", "P3": "#dcf0e1", "P4": "#ebe4f7"}
PROC_NAME = {"P1": "Boiler", "P2": "Turbine", "P3": "Water treatment", "P4": "HIL simulation"}
TEAL, TEAL_L = "#138a8a", "#d6f0ef"
BLUE, BLUE_L = "#2a6fd6", "#dbe7fb"
ORANGE, ORANGE_L = "#e8902a", "#fcebd6"
GREEN, GREEN_L = "#3f9c55", "#dcf0e1"
PURPLE, PURPLE_L = "#8b6fc4", "#ebe4f7"
RED, RED_L = "#c8433f", "#f8dedd"
GREY_L = "#f3f4f6"


def save(fig, F, name):
    fig.savefig(f"{F}/{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(f"{F}/{name}.tif", dpi=300, bbox_inches="tight", facecolor="white",
                pil_kwargs={"compression": "tiff_lzw"})
    plt.close(fig)


class Canvas:
    """Drawing helper in a top-left coordinate system (x right, y down)."""

    def __init__(self, w, h, inch_h):
        self.W, self.H = w, h
        self.fig, self.ax = plt.subplots(figsize=(W_IN, inch_h))
        self.ax.set_xlim(0, w); self.ax.set_ylim(0, h); self.ax.axis("off")
        self.fig.subplots_adjust(0, 0, 1, 1)

    def Y(self, y):
        return self.H - y

    def box(self, x, y, w, h, title=None, lines=(), fc="white", ec=INK, lw=1.0, tsize=8, lsize=7, tcolor=INK, r=7, bold=True):
        self.ax.add_patch(FancyBboxPatch((x, self.Y(y + h)), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                                         fc=fc, ec=ec, lw=lw, zorder=2))
        n = (1 if title else 0) + len(lines)
        top = y + h / 2 - (n - 1) * 6.5
        k = 0
        if title:
            self.ax.text(x + w / 2, self.Y(top), title, ha="center", va="center", fontsize=tsize,
                         weight="bold" if bold else "normal", color=tcolor, zorder=3); k = 1
        for j, l in enumerate(lines):
            self.ax.text(x + w / 2, self.Y(top + 13 * (j + k)), l, ha="center", va="center", fontsize=lsize, color=MUTED, zorder=3)

    def band(self, x, y, w, h, fc, label=None, lc=MUTED):
        self.ax.add_patch(FancyBboxPatch((x, self.Y(y + h)), w, h, boxstyle="round,pad=0,rounding_size=9", fc=fc, ec="none", zorder=0))
        if label:
            self.ax.text(x + 9, self.Y(y + 11), label, fontsize=6.8, weight="bold", color=lc, va="center", zorder=1)

    def arrow(self, x1, y1, x2, y2, color=INK, lw=1.1, style="-|>", ls="-"):
        self.ax.annotate("", xy=(x2, self.Y(y2)), xytext=(x1, self.Y(y1)), zorder=4,
                         arrowprops=dict(arrowstyle=style, color=color, lw=lw, linestyle=ls, shrinkA=0, shrinkB=0))

    def line(self, xs, ys, color=INK, lw=1.1):
        self.ax.plot(xs, [self.Y(v) for v in ys], color=color, lw=lw, zorder=1, solid_capstyle="round")

    def text(self, x, y, s, **kw):
        kw.setdefault("fontsize", 7); kw.setdefault("color", MUTED); kw.setdefault("ha", "center"); kw.setdefault("va", "center")
        self.ax.text(x, self.Y(y), s, zorder=5, **kw)


# ----------------------------------------------------------------------------------------------- Figure 1
def fig_pipeline(R, F):
    a = json.load(open(f"{R}/dataset_audit.json"))
    c = Canvas(760, 400, 3.75)
    c.band(6, 6, 748, 108, TEAL_L, "1  DATA (HAI 21.03, 1 Hz, 57 sensors)", TEAL)
    c.band(6, 124, 748, 108, BLUE_L, "2  MODELS AND ATTRIBUTION (normal data only)", BLUE)
    c.band(6, 242, 748, 152, GREY_L, "3  EVALUATION AGAINST RECORDED ATTACK LABELS", INK)
    W, xs = 222, [24, 269, 514]
    c.box(xs[0], 30, W, 70, "Training files", ["normal operation only", f"{a['fit_rows']:,} fitting rows"], ec=TEAL, lw=1.4)
    c.box(xs[1], 30, W, 70, "Calibration tail", ["last 20% of final training file", f"{a['cal_rows']:,} rows"], ec=TEAL, lw=1.4)
    c.box(xs[2], 30, W, 70, "Test files", [f"{a['test_rows']:,} rows", f"{a['events']} real attacks, labels held back"], ec=TEAL, lw=1.4)
    c.box(xs[0], 148, W, 70, "Five detectors", ["PCA, MLP autoencoder, isolation", "forest, Z-max, USAD (deep)"], ec=BLUE, lw=1.4)
    c.box(xs[1], 148, W, 70, "Score and alarm", ["causal 60-s mean;", "threshold = q-quantile (q = 0.999)"], ec=BLUE, lw=1.4)
    c.box(xs[2], 148, W, 70, "Process attribution", ["exact group Shapley (16 evaluations)", "vs residual, Z-mean, Kernel SHAP"], ec=ORANGE, lw=1.8, fc="#fff7ee")
    for x in xs:
        c.arrow(x + W / 2, 100, x + W / 2, 148, color=MUTED)
    c.arrow(xs[0] + W, 183, xs[1], 183); c.arrow(xs[1] + W, 183, xs[2], 183)
    outs = [("Detection", ["point F1, AUC-PR, event recall,", "delay, false alarms per hour", "Tables 3, 10"], GREEN, GREEN_L),
            ("Localization (RQ1, RQ2)", ["top process vs attacked process;", "stealth tiers, oracle, baseline", "Figs 5, 6, 9, 11; Tables 4-6, 11-13"], ORANGE, ORANGE_L),
            ("Trust gate (RQ3)", ["alert after 10 stable seconds", "vs duration filter", "Fig. 7; Table 7"], PURPLE, PURPLE_L)]
    for x, (t, l, ec, fc) in zip(xs, outs):
        c.box(x, 286, W, 92, t, l, fc=fc, ec=ec, lw=1.6)
    c.line([xs[1] + W / 2, xs[1] + W / 2], [218, 266], color=MUTED)
    c.line([xs[0] + W / 2, xs[2] + W / 2], [266, 266], color=MUTED)
    c.line([xs[2] + W / 2, xs[2] + W / 2], [218, 266], color=MUTED)
    for x in xs:
        c.arrow(x + W / 2, 266, x + W / 2, 286, color=MUTED)
    save(c.fig, F, "Fig1_pipeline")


# ----------------------------------------------------------------------------------------------- Figure 2
def fig_hierarchy(R, R22, F):
    a = json.load(open(f"{R}/dataset_audit.json")); b = json.load(open(f"{R22}/audit22.json"))
    loops = b["loop_groups"]
    c = Canvas(760, 330, 3.2)
    c.box(190, 8, 380, 40, "HAI testbed (boiler, turbine, water treatment, HIL)", fc=GREY_L, ec=INK, tsize=7.5)
    c.text(14, 28, "Level", ha="left", weight="bold", color=INK, fontsize=7)
    c.text(14, 96, "Process", ha="left", weight="bold", color=INK, fontsize=7.5)
    c.text(14, 108, "4 groups", ha="left", fontsize=6.5)
    c.text(14, 196, "Control loop", ha="left", weight="bold", color=INK, fontsize=7.5)
    c.text(14, 208, "11 groups", ha="left", fontsize=6.5)
    c.text(14, 286, "Data point", ha="left", weight="bold", color=INK, fontsize=7.5)
    c.text(14, 298, f"{b['features_used']} points", ha="left", fontsize=6.5)
    by_p = {p: [L for L in loops if L.startswith(p)] for p in PROC}
    wt = {L: (1.6 if len(by_p[L[:2]]) == 1 else 1.0) for L in loops}
    x0 = 96; unit = (752 - x0) / sum(wt.values())
    c.text(14, 120, "sensors:", ha="left", fontsize=6)
    c.text(14, 130, "21.03 / 22.04", ha="left", fontsize=6)
    two = {"P3": ["Water", "treatment"], "P4": ["HIL", "simulation"]}
    x = x0
    for p in PROC:
        w = sum(wt[L] for L in by_p[p]) * unit
        px = x
        if p in two:
            c.box(px + 3, 68, w - 6, 62, p, two[p] + [f"{a['group_sizes'][p]} / {b['process_groups'][p]}"],
                  fc=PROC_LIGHT[p], ec=PROC[p], lw=1.6, tsize=7.5, lsize=6.2)
        else:
            c.box(px + 3, 68, w - 6, 62, f"{p}  {PROC_NAME[p]}", [f"{a['group_sizes'][p]} / {b['process_groups'][p]} sensors"],
                  fc=PROC_LIGHT[p], ec=PROC[p], lw=1.6, tsize=7.5, lsize=6.4)
        c.arrow(380, 48, px + w / 2, 68, color=MUTED, lw=0.8)
        cx = px
        for L in by_p[p]:
            lw_ = wt[L] * unit
            c.box(cx + 3, 170, lw_ - 6, 48, L, [f"{loops[L]} points"], fc="white", ec=PROC[p], lw=1.3, tsize=6.2, lsize=5.9)
            c.arrow(px + w / 2, 130, cx + lw_ / 2, 170, color=PROC[p], lw=0.7)
            m = min(loops[L], 6)
            for k in range(m):
                dx = cx + lw_ / 2 + (k - (m - 1) / 2) * 7
                c.ax.add_patch(Circle((dx, c.Y(272)), 2.5, fc=PROC[p], ec="none", zorder=3))
            if loops[L] > 6:
                c.text(cx + lw_ / 2, 285, f"+{loops[L] - 6}", fontsize=5.8)
            c.line([cx + lw_ / 2, cx + lw_ / 2], [218, 266], color=PROC[p], lw=0.5)
            cx += lw_
        x += w
    c.text(380, 318, "Localization is scored at each level: correct process, correct control loop, and target point in the top 1, 3 or 5",
           fontsize=6.8, color=INK)
    save(c.fig, F, "Fig2_hierarchy")


# ----------------------------------------------------------------------------------------------- Figure 3
def fig_shapley(R, F):
    cs = pd.read_csv(f"{R}/E7_case_studies.csv")
    ev = 3 if (cs.event == 3).any() else int(cs.event.iloc[0])
    sub = cs[cs.event == ev].set_index("method"); target = sub.targets.iloc[0]
    fig = plt.figure(figsize=(W_IN, 3.3))
    # panel a: coalition lattice
    ax = fig.add_axes([0.0, 0.0, 0.65, 1.0]); ax.set_xlim(0, 500); ax.set_ylim(0, 300); ax.axis("off")
    ax.text(4, 288, "a", fontsize=10, weight="bold", color=INK)
    ax.text(22, 288, "Coalitions S of the four processes: 2⁴ = 16 detector evaluations per second", fontsize=7.5, color=INK, va="center")
    import itertools
    G = list(PROC)
    levels = [list(itertools.combinations(G, k)) for k in range(5)]
    pos = {}
    for k, combs in enumerate(levels):
        y = 255 - k * 52
        for i, S in enumerate(combs):
            x = 300 + (i - (len(combs) - 1) / 2) * 66
            pos[S] = (x, y)
    for k in range(4):
        for S in levels[k]:
            for T in levels[k + 1]:
                if set(S) <= set(T):
                    g = (set(T) - set(S)).pop()
                    ax.plot([pos[S][0], pos[T][0]], [pos[S][1] - 9, pos[T][1] + 9], color=PROC[g], lw=0.6, alpha=0.55, zorder=1)
    for S, (x, y) in pos.items():
        ax.add_patch(FancyBboxPatch((x - 27, y - 9), 54, 18, boxstyle="round,pad=0,rounding_size=4", fc="white", ec="#9ca3af", lw=0.7, zorder=2))
        for j, g in enumerate(G):
            ax.add_patch(plt.Rectangle((x - 22 + j * 11.5, y - 5), 9, 10, fc=PROC[g] if g in S else "#e5e7eb", ec="none", zorder=3))
    ax.text(235 + 2.5 * 70 + 14, 255, "v(∅): every process\nat training median", fontsize=6.3, color=MUTED, va="center", ha="left")
    ax.text(300 + 32, 47, "v(G) = s(x): observed sample", fontsize=6.3, color=MUTED, va="center", ha="left")
    for j, g in enumerate(G):
        ax.add_patch(plt.Rectangle((12, 200 - j * 18), 9, 10, fc=PROC[g], ec="none"))
        ax.text(26, 205 - j * 18, f"{g} observed", fontsize=6.5, va="center", color=INK)
    ax.add_patch(plt.Rectangle((12, 128), 9, 10, fc="#e5e7eb", ec="none"))
    ax.text(26, 133, "replaced by median", fontsize=6.5, va="center", color=INK)
    ax.text(12, 95, "φ(g) = weighted mean of\nv(S ∪ {g}) − v(S)\nover the 8 coalitions\nwithout g (colored\nedges, one colour\nper process g)", fontsize=6.4, color=INK, va="top", linespacing=1.35)
    # panel b: real attribution for one event
    ax2 = fig.add_axes([0.72, 0.14, 0.27, 0.62])
    meths = ["Residual", "Group Shapley"]
    w = 0.38
    for i, m in enumerate(meths):
        vals = [sub.loc[m, f"share_{g}"] for g in G]
        for j, g in enumerate(G):
            ax2.bar(j + (i - 0.5) * w, vals[j], w * 0.92, color=PROC[g] if m == "Group Shapley" else PROC_LIGHT[g],
                    ec=PROC[g], lw=0.8, hatch=None if m == "Group Shapley" else "////", zorder=3)
    ax2.set_xticks(range(4)); ax2.set_xticklabels(G, fontsize=7.5)
    ax2.set_ylabel("Mean attribution share", fontsize=7); ax2.tick_params(labelsize=6.5)
    ax2.set_ylim(0, 0.8); ax2.grid(axis="y", color="#eeeeee", zorder=0)
    for s in ("top", "right"):
        ax2.spines[s].set_visible(False)
    from matplotlib.patches import Patch
    ax2.legend(handles=[Patch(fc="white", ec=INK, hatch="////", label="Residual"), Patch(fc=INK, label="Group Shapley")],
               fontsize=6.3, frameon=False, loc="upper right")
    fig.text(0.675, 0.94, "b", fontsize=10, weight="bold", color=INK)
    fig.text(0.70, 0.94, f"Event {ev}: {target} attacked (PCA detector)", fontsize=7.3, color=INK, va="center")
    fig.text(0.70, 0.865, "residual points to P3,\ngroup Shapley to P2", fontsize=6.6, color=MUTED, va="center")
    save(fig, F, "Fig3_group_shapley")


# ----------------------------------------------------------------------------------------------- Figure 4
def fig_trust_gate(F):
    c = Canvas(760, 372, 3.5)
    c.band(6, 6, 478, 360, PURPLE_L, "TRUST GATE (this study)", PURPLE)
    c.band(494, 6, 260, 360, GREY_L, "DURATION FILTER (baseline)", INK)
    c.box(26, 30, 218, 44, "Alarm second t", ["smoothed score above threshold"], ec=INK, lw=1.2)
    c.box(26, 98, 218, 52, "Five noisy copies of the input", ["Gaussian noise, \u03c3 \u2208 {0.05, 0.2, 0.5}", "(z-units)"], ec=BLUE, fc=BLUE_L, lw=1.3)
    c.box(26, 174, 218, 44, "Residual attribution", ["top process of each copy"], ec=ORANGE, fc=ORANGE_L, lw=1.3)
    cx, cy = 135, 264
    c.ax.add_patch(plt.Polygon([(cx, c.Y(cy - 36)), (cx + 96, c.Y(cy)), (cx, c.Y(cy + 36)), (cx - 96, c.Y(cy))], fc="white", ec=PURPLE, lw=1.5, zorder=2))
    c.text(cx, cy - 12, "Original top process", fontsize=7, color=INK, weight="bold")
    c.text(cx, cy, "kept in \u2265 4 of 5", fontsize=7, color=INK, weight="bold")
    c.text(cx, cy + 12, "copies?", fontsize=7, color=INK, weight="bold")
    for y1, y2 in [(74, 98), (150, 174), (218, 228)]:
        c.arrow(135, y1, 135, y2)
    c.box(290, 26, 176, 52, "Raise alert", ["once the segment holds", "10 stable seconds"], ec=GREEN, fc="white", lw=1.8, tcolor=GREEN)
    c.box(290, 110, 176, 52, "Stable count + 1", ["stable seconds so far", "in this alarm segment"], ec=GREEN, fc=GREEN_L, lw=1.3)
    c.box(290, 296, 176, 52, "Not counted", ["count unchanged; check", "the next alarm second"], ec=RED, fc=RED_L, lw=1.3)
    c.line([231, 378], [264, 264], color=GREEN); c.arrow(378, 264, 378, 162, color=GREEN)
    c.text(262, 255, "yes", color=GREEN, weight="bold")
    c.arrow(378, 110, 378, 74, color=GREEN)
    c.line([135, 135], [300, 322], color=RED); c.arrow(135, 322, 290, 322, color=RED)
    c.text(150, 312, "no", color=RED, weight="bold")
    c.box(526, 30, 196, 44, "Alarm second t", ["smoothed score above threshold"], ec=INK, lw=1.2)
    c.box(526, 130, 196, 52, "Consecutive count + 1", ["no use of the explanation"], ec=MUTED, fc="white", lw=1.2)
    c.box(526, 236, 196, 52, "Raise alert", ["after k seconds,", "k \u2208 {1, 5, 10, \u2026, 120}"], ec=INK, fc="white", lw=1.6)
    c.arrow(624, 74, 624, 130, color=MUTED); c.arrow(624, 182, 624, 236, color=MUTED)
    c.text(624, 318, "Compared on the full recall vs", fontsize=6.6)
    c.text(624, 330, "false-alarm curve (Figure 7)", fontsize=6.6)
    save(c.fig, F, "Fig4_trust_gate_flow")


# ----------------------------------------------------------------------------------------------- Figure 12
def fig_architecture(F):
    c = Canvas(760, 372, 3.6)
    c.band(8, 6, 744, 112, TEAL_L, "OFFLINE  (normal operation only)", TEAL)
    c.band(8, 128, 744, 128, BLUE_L, "ONLINE  (every second)", BLUE)
    c.band(8, 266, 744, 100, GREY_L, "OPERATOR AND RESPONSE", INK)
    W4, G4 = 168, 18
    xs4 = [22 + k * (W4 + G4) for k in range(4)]
    off = [("Historian data", ["1-Hz sensor and actuator", "tags (OPC, Modbus)"]), ("Fit detector", ["standardize; any model:", "PCA, AE, USAD, IForest"]),
           ("Calibrate", ["threshold q on a held-out", "normal tail"]), ("Process map", ["sensor → process / loop", "(P1 … P4)"])]
    for x, (t, l) in zip(xs4, off):
        c.box(x, 30, W4, 70, t, l, ec=TEAL, lw=1.4)
    for k in range(3):
        c.arrow(xs4[k] + W4, 65, xs4[k + 1], 65, color=TEAL)
    W5, G5 = 132, 15
    xs5 = [22 + k * (W5 + G5) for k in range(5)]
    on = [("Live tags", ["buffered and", "standardized"], INK, "white"), ("Detector score", ["causal 60-s mean,", "compared with q"], BLUE, "white"),
          ("Group Shapley", ["16 evaluations;", "top = location"], ORANGE, ORANGE_L), ("Trust gate", ["alert after 10", "stable seconds"], PURPLE, PURPLE_L),
          ("Operator alert", ["process, share,", "stability"], GREEN, GREEN_L)]
    for x, (t, l, ec, fc) in zip(xs5, on):
        c.box(x, 152, W5, 70, t, l, ec=ec, fc=fc, lw=1.6)
    for k in range(4):
        c.arrow(xs5[k] + W5, 187, xs5[k + 1], 187)
    c.text(xs5[1] + W5 / 2, 236, "below threshold: no alert", fontsize=6.3)
    c.text(xs5[3] + W5 / 2, 236, "unstable: hold and re-check", fontsize=6.3)
    c.arrow(xs4[1] + W4 / 2, 100, xs5[1] + W5 / 2, 152, color=TEAL, ls="--")
    c.arrow(xs4[2] + W4 / 2, 100, xs5[1] + W5 * 0.8, 152, color=TEAL, ls="--")
    c.arrow(xs4[3] + W4 / 2, 100, xs5[2] + W5 / 2, 152, color=TEAL, ls="--")
    c.box(150, 290, 210, 60, "Inspect the named process", ["check its loops and points", "(hierarchical ranking)"], ec=INK, lw=1.2)
    c.box(400, 290, 210, 60, "Log for audit", ["alert, attribution share,", "stability, timestamps"], ec=INK, lw=1.2)
    ax_ = xs5[4] + W5 / 2
    c.line([ax_, ax_], [222, 276], color=GREEN); c.line([255, ax_], [276, 276], color=GREEN)
    c.arrow(255, 276, 255, 290, color=GREEN); c.arrow(505, 276, 505, 290, color=GREEN)
    save(c.fig, F, "Fig12_architecture")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--results22", default="results22")
    a = ap.parse_args()
    F = f"{a.results}/figures"; os.makedirs(F, exist_ok=True)
    fig_pipeline(a.results, F)
    if os.path.exists(f"{a.results22}/audit22.json"):
        fig_hierarchy(a.results, a.results22, F)
    fig_shapley(a.results, F)
    fig_trust_gate(F)
    fig_architecture(F)
    print("diagrams ok ->", F)


if __name__ == "__main__":
    main()


In [ ]:
!python diagrams.py --results results --results22 results22
from IPython.display import Image, display
for f in ['Fig1_pipeline','Fig2_hierarchy','Fig3_group_shapley','Fig4_trust_gate_flow','Fig12_architecture']:
    display(Image(f'results/figures/{f}.png', width=900))
!zip -qr results.zip results

## 10. Deep detector (USAD) and Kernel SHAP baseline

Adds USAD (PyTorch) to the full protocol and compares per-sensor Kernel SHAP with exact group Shapley on identical seconds. About 30 min (HAI 21.03) and 1.5 h (HAI 22.04) on a 2-core CPU; faster on a GPU runtime is not needed.

In [ ]:
%%writefile extend.py
"""
Two additions requested by the planned reviewer checklist (run after fdi_pipeline.py):

  E10  USAD (Audibert et al., 2020), a deep adversarially trained autoencoder on
       10-second windows, run through exactly the same detection, localization,
       stealth and trust-gate protocol as the four detectors in fdi_pipeline.py.
       Rows are written with detector = "USAD" in E10_*.csv (same schema as E1-E4).
  E11  Per-sensor Kernel SHAP (Lundberg and Lee, 2017), summed to processes, compared
       with exact group Shapley on the SAME sampled alarm seconds of every detected
       event, for every detector and seed. Also times both attributions.

Usage:
    python extend.py --data hai/hai-21.03 --results results [--seeds 5]
"""
import argparse, json, os, sys, time
from itertools import combinations
from math import comb
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fdi_pipeline as fp

WIN = 10              # USAD window length (s)
KS_SAMPLES = 1024     # Kernel SHAP coalitions per explained second
KS_PER_EVENT = 20     # alarm seconds explained per detected event (evenly spaced)


# ======================================================================= USAD
def lagged(X, starts, win=WIN, rows=None):
    """Row t -> concatenation of rows t, t-1, ..., t-win+1 (lag-major), padded at segment starts.
    `starts` = first row index of every contiguous segment (file); `rows` = optional subset of t."""
    n, d = X.shape
    seg_start = np.zeros(n, dtype=np.int64)
    bounds = list(starts) + [n]
    for a, b in zip(bounds[:-1], bounds[1:]):
        seg_start[a:b] = a
    rows = np.arange(n) if rows is None else np.asarray(rows)
    out = np.empty((len(rows), d * win), dtype=np.float32)
    for k in range(win):
        idx = np.maximum(rows - k, seg_start[rows])
        out[:, k * d:(k + 1) * d] = X[idx]
    return out


class USAD(fp.Detector):
    """USAD with one encoder and two decoders, trained with the two-phase adversarial
    objective of Audibert et al. (2020). Input = 10-s window (lag features), min-max scaled
    to [0, 1] on the training windows; decoders end in a sigmoid (reference implementation).
    score = mean over window features of 0.5*(x-AE1)^2 + 0.5*(x-AE2(AE1))^2, so that
    score = mean(resid^2) as for the other reconstruction detectors."""
    name = "USAD"

    def __init__(self, seed=0, latent=40, epochs=20, n_train=60_000):
        self.seed, self.latent, self.epochs, self.n_train = seed, latent, epochs, n_train

    def fit(self, X):
        import torch, torch.nn as nn
        torch.manual_seed(self.seed); np.random.seed(self.seed); torch.set_num_threads(os.cpu_count() or 2)
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(len(X), size=min(self.n_train, len(X)), replace=False)
        # min-max scaling to [0, 1] on the training windows, with sigmoid decoder outputs,
        # as in the reference USAD implementation
        self.lo = X[idx].min(axis=0); self.rng = np.maximum(X[idx].max(axis=0) - self.lo, 1e-6)
        Xt = torch.tensor(self._mm(X[idx]))
        d = X.shape[1]; h1, h2 = d // 2, d // 4
        enc = nn.Sequential(nn.Linear(d, h1), nn.ReLU(), nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, self.latent), nn.ReLU())
        dec = lambda: nn.Sequential(nn.Linear(self.latent, h2), nn.ReLU(), nn.Linear(h2, h1), nn.ReLU(), nn.Linear(h1, d), nn.Sigmoid())
        self.enc, self.d1, self.d2 = enc, dec(), dec()
        o1 = torch.optim.Adam(list(enc.parameters()) + list(self.d1.parameters()), lr=1e-3)
        o2 = torch.optim.Adam(list(enc.parameters()) + list(self.d2.parameters()), lr=1e-3)
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(1, self.epochs + 1):
            perm = torch.randperm(len(Xt), generator=g)
            for b in range(0, len(Xt), 512):
                w = Xt[perm[b:b + 512]]
                z = enc(w); w1 = self.d1(z); w2 = self.d2(z); w3 = self.d2(enc(w1))
                l1 = (1 / ep) * ((w - w1) ** 2).mean() + (1 - 1 / ep) * ((w - w3) ** 2).mean()
                o1.zero_grad(); l1.backward(); o1.step()
                z = enc(w); w1 = self.d1(z); w2 = self.d2(z); w3 = self.d2(enc(w1))
                l2 = (1 / ep) * ((w - w2) ** 2).mean() - (1 - 1 / ep) * ((w - w3) ** 2).mean()
                o2.zero_grad(); l2.backward(); o2.step()
        for m in (self.enc, self.d1, self.d2):
            m.eval()
        return self

    def _mm(self, X):
        return ((X - self.lo) / self.rng).astype(np.float32)

    def resid(self, X):
        import torch
        out = np.empty_like(X, dtype=np.float32)
        with torch.no_grad():
            for b in range(0, len(X), 8192):
                w = torch.tensor(self._mm(X[b:b + 8192])); w1 = self.d1(self.enc(w)); w3 = self.d2(self.enc(w1))
                out[b:b + 8192] = torch.sqrt(0.5 * (w - w1) ** 2 + 0.5 * (w - w3) ** 2).numpy()
        return out

    def score(self, X):
        return np.mean(self.resid(X) ** 2, axis=1)


# ================================================================ Kernel SHAP
def kernel_shap(score, X, players, base, n_samples=KS_SAMPLES, seed=0):
    """Kernel SHAP (Lundberg and Lee, 2017) with a single reference point `base`.
    players: list of column-index arrays (one per player). Coalition sizes are sampled
    in proportion to the Shapley kernel, in complementary pairs, and the efficiency
    constraint sum(phi) = f(x) - f(base) is imposed exactly. Returns (n, M)."""
    rng = np.random.default_rng(seed)
    M = len(players)
    sizes = np.arange(1, M)
    w_size = (M - 1) / (sizes * (M - sizes)); w_size /= w_size.sum()
    half = n_samples // 2
    Z = np.zeros((2 * half, M), dtype=bool)
    s = rng.choice(sizes, size=half, p=w_size)
    for i, k in enumerate(s):
        on = rng.choice(M, size=k, replace=False)
        Z[2 * i, on] = True; Z[2 * i + 1] = ~Z[2 * i]
    col_of = np.zeros(sum(len(p) for p in players), dtype=np.int64)
    for j, p in enumerate(players):
        col_of[p] = j
    f0 = score(base[None, :])[0]
    fx = score(X)
    out = np.zeros((len(X), M))
    Zf = Z.astype(np.float64)
    A = Zf[:, :-1] - Zf[:, [-1]]
    for i, x in enumerate(X):
        Xs = np.where(Z[:, col_of], x[None, :], base[None, :]).astype(np.float32)
        y = score(Xs) - f0 - Zf[:, -1] * (fx[i] - f0)
        phi, *_ = np.linalg.lstsq(A, y, rcond=None)      # uniform weights: sizes already sampled by kernel
        out[i, :-1] = phi; out[i, -1] = (fx[i] - f0) - phi.sum()
    return out


def sample_seconds(idx, k=KS_PER_EVENT):
    if len(idx) <= k:
        return idx
    return idx[np.linspace(0, len(idx) - 1, k).round().astype(int)]


def validate_kernel_shap(det, X, gidx, base):
    """Kernel SHAP with the 4 processes as players must reproduce exact group Shapley."""
    players = [gidx[g] for g in fp.GROUPS]
    ks = kernel_shap(det.score, X, players, base, n_samples=4096)
    ex = fp.attr_group_shapley(det, X, gidx, base)
    return float(np.max(np.abs(ks - ex)) / (np.max(np.abs(ex)) + 1e-12))


# ======================================================== shared protocol run
def run_protocol(D, seed, Xfit, Xcal, Xte, base, gidx, y, files, ev, gate=True):
    """Same steps as fdi_pipeline.main for one detector and seed (detection,
    localization, stealth tiers, trust gate). Returns row lists and the fitted detector."""
    in_attack = y == 1
    det_rows, loc_rows, tier_rows, gate_rows, perev_rows, curve_rows = [], [], [], [], [], []
    t1 = time.perf_counter(); det = D(seed).fit(Xfit); fit_s = time.perf_counter() - t1
    cal_s = fp.smooth(det.score(Xcal), np.zeros(len(Xcal)))
    thr = float(np.quantile(cal_s, fp.THR_Q))
    s = fp.smooth(det.score(Xte), files); alarm = s > thr
    m, detected = fp.evaluate(alarm, y, files, ev, s)
    det_rows.append(dict(detector=D.name, seed=seed, threshold=thr, fit_s=fit_s, **m))
    cal_r = det.resid(Xcal)
    cal_sd = (np.std(cal_r, axis=0) + 1e-6) if cal_r is not None else None
    methods = {"Z-mean": lambda X: fp.attr_z(X[:, :len(base_cur)], gidx_cur),
               "GroupShapley": lambda X: fp.attr_group_shapley(det, X, gidx, base)}
    if cal_sd is not None:
        methods["Residual"] = lambda X: fp.attr_residual(det, X, gidx, cal_sd)
    for mname, fn in methods.items():
        hits = []
        for _, e in ev.iterrows():
            allidx = np.arange(e.start, e.end + 1)
            top_or = fp.GROUPS[int(np.argmax(fn(Xte[allidx]).mean(axis=0)))]
            hit_or = float(top_or in e.targets.split("+"))
            idx = allidx[alarm[allidx]]
            if len(idx) == 0:
                hits.append(np.nan); top = None; hit = np.nan
            else:
                top = fp.GROUPS[int(np.argmax(fn(Xte[idx]).mean(axis=0)))]
                hit = float(top in e.targets.split("+")); hits.append(hit)
            perev_rows.append(dict(detector=D.name, seed=seed, method=mname, event=e.event, targets=e.targets,
                                   single=e.single, top=top, hit=hit, top_oracle=top_or, hit_oracle=hit_or,
                                   magnitude_z=e.magnitude_z, stealth_tier=str(e.stealth_tier)))
        hits = np.array(hits); sing = ev.single.to_numpy(); lo, hi = fp.boot_ci(hits[sing])
        loc_rows.append(dict(detector=D.name, seed=seed, method=mname, localized_events=int(np.nansum(hits[sing])),
                             detected_single=int((~np.isnan(hits[sing])).sum()), hit_rate_single=float(np.nanmean(hits[sing])),
                             ci_low=lo, ci_high=hi,
                             hit_rate_multi=float(np.nanmean(hits[~sing])) if (~np.isnan(hits[~sing])).any() else np.nan))
        for tier in ["high stealth", "medium", "low stealth"]:
            mk = (ev.stealth_tier == tier).to_numpy()
            tier_rows.append(dict(detector=D.name, seed=seed, method=mname, tier=tier, events=int(mk.sum()),
                                  detection_rate=float(detected[mk].mean()),
                                  localization_rate_all=float(np.nan_to_num(hits[mk]).mean()),
                                  localization_rate_detected=float(np.nanmean(hits[mk])) if (~np.isnan(hits[mk])).any() else np.nan))
    if gate and cal_sd is not None:
        for q in fp.GATE_QS:
            al_q = s > float(np.quantile(cal_s, q)); aidx = np.flatnonzero(al_q)
            top0 = np.argmax(fp.attr_residual(det, Xte[aidx], gidx, cal_sd), axis=1)
            for sd_ in fp.GATE_SDS:
                rng = np.random.default_rng(seed); same = np.zeros(len(aidx))
                for _ in range(fp.NOISE_REPS):
                    Xn = Xte[aidx] + rng.normal(0, sd_, size=Xte[aidx].shape).astype(np.float32)
                    same += np.argmax(fp.attr_residual(det, Xn, gidx, cal_sd), axis=1) == top0
                stable = np.zeros(len(y), bool); stable[aidx] = same >= fp.NOISE_REPS - 1
                pols = [("trust gate", fp.trust_gate(al_q, stable, files, fp.GATE_K))]
                if sd_ == fp.GATE_SDS[0]:
                    pols = [("no gate", al_q), ("duration filter", fp.duration_filter(al_q, files, fp.GATE_K))] + pols
                for policy, al in pols:
                    mm, _ = fp.evaluate(al, y, files, ev)
                    gate_rows.append(dict(detector=D.name, seed=seed, thr_quantile=q,
                                          noise_sd=sd_ if policy == "trust gate" else np.nan, policy=policy,
                                          stable_share_attack=float(stable[aidx][in_attack[aidx]].mean()),
                                          stable_share_normal=float(stable[aidx][~in_attack[aidx]].mean()) if (~in_attack[aidx]).any() else np.nan,
                                          **mm))
            for k in [1, 5, 10, 20, 30, 60, 90, 120]:
                mk_, _ = fp.evaluate(fp.duration_filter(al_q, files, k), y, files, ev)
                curve_rows.append(dict(detector=D.name, seed=seed, thr_quantile=q, k=k, event_recall=mk_["event_recall"],
                                       false_alarms_per_hour=mk_["false_alarms_per_hour"], median_delay_s=mk_["median_delay_s"]))
    return det, alarm, (det_rows, loc_rows, tier_rows, gate_rows, perev_rows, curve_rows)


# globals used by the Z-mean lambda (current-row z-scores, also for lagged USAD input)
base_cur, gidx_cur = None, None


def main():
    global base_cur, gidx_cur
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--results", default="results")
    ap.add_argument("--seeds", type=int, default=len(fp.SEEDS))
    ap.add_argument("--skip_usad", action="store_true")
    a = ap.parse_args(); R = a.results; t0 = time.time()

    tr, te = fp.load(a.data)
    feats, gidx, Xfit, Xcal, Xte, base = fp.prepare(tr, te)
    base_cur, gidx_cur = base, gidx
    ev = fp.events(te); evs = pd.read_csv(f"{R}/events.csv")
    ev["magnitude_z"] = evs.magnitude_z.values; ev["stealth_tier"] = evs.stealth_tier.values
    y = te["attack"].to_numpy().astype(int); files = te["file"].to_numpy()
    d = Xfit.shape[1]; seeds = fp.SEEDS[:a.seeds]

    # ---------------------------------------------------------------- E10 USAD
    n1, n2 = len(tr[0]), len(tr[1])
    fit_starts = [0, n1, n1 + n2]
    del tr
    import gc; gc.collect()
    te_starts = [int(np.flatnonzero(files == f)[0]) for f in np.unique(files)]
    # USAD trains on 60,000 windows per seed; build windows only for a fixed pool of 200,000 fitting rows
    pool = np.sort(np.random.default_rng(0).choice(len(Xfit), size=200_000, replace=False))
    Lfit, Lcal, Lte = lagged(Xfit, fit_starts, rows=pool), lagged(Xcal, [0]), lagged(Xte, te_starts)
    gidx_L = {g: np.concatenate([gidx[g] + k * d for k in range(WIN)]) for g in fp.GROUPS}
    base_L = np.tile(base, WIN).astype(np.float32)
    usad = {}
    if not a.skip_usad:
        rows = [[] for _ in range(6)]
        for seed in seeds:
            det, alarm, out = run_protocol(USAD, seed, Lfit, Lcal, Lte, base_L, gidx_L, y, files, ev)
            usad[seed] = (det, alarm)
            for r, o in zip(rows, out):
                r += o
            print(f"[{time.time()-t0:6.0f}s] USAD seed {seed}: F1={out[0][0]['f1']:.3f} "
                  f"event recall={out[0][0]['event_recall']:.2f} FA/h={out[0][0]['false_alarms_per_hour']:.3f}", flush=True)
        for name, r in zip(["E1_detection", "E2_localization", "E3_stealth_tiers", "E4_trust_gate",
                            "E2_localization_per_event", "E4_duration_curve"], rows):
            pd.DataFrame(r).to_csv(f"{R}/E10_{name}.csv", index=False)

    # ---------------------------------------------------------- E11 Kernel SHAP
    # validation on the PCA detector: 4-player Kernel SHAP vs exact group Shapley
    pca = fp.PCARecon(42).fit(Xfit)
    att = np.flatnonzero(y == 1)
    val_err = validate_kernel_shap(pca, Xte[att[::200][:40]], gidx, base)
    print(f"Kernel SHAP vs exact (4 players), max relative error = {val_err:.4f}", flush=True)

    sensors = [np.array([j]) for j in range(d)]
    sensors_L = [np.array([j + k * d for k in range(WIN)]) for j in range(d)]
    proc_of = np.array([next(g for g in fp.GROUPS if j in gidx[g]) for j in range(d)])
    rows, cost = [], []
    dets = [(fp.PCARecon, seeds[:1]), (fp.MLPAE, seeds), (fp.IForest, seeds), (fp.ZMax, seeds[:1])]
    if usad:
        dets.append((USAD, seeds))
    for D, sd_list in dets:
        for seed in sd_list:
            if D is USAD:
                det, alarm = usad[seed]; XT, B, G, P = Lte, base_L, gidx_L, sensors_L
                cal_s = None
            else:
                det = D(seed).fit(Xfit)
                cal_s = fp.smooth(det.score(Xcal), np.zeros(len(Xcal)))
                alarm = fp.smooth(det.score(Xte), files) > float(np.quantile(cal_s, fp.THR_Q))
                XT, B, G, P = Xte, base, gidx, sensors
            t_ks = t_gs = 0.0; n_expl = 0
            for _, e in ev.iterrows():
                idx = np.arange(e.start, e.end + 1); idx = idx[alarm[idx]]
                if len(idx) == 0:
                    continue
                sel = sample_seconds(idx)
                t1 = time.perf_counter(); phi = kernel_shap(det.score, XT[sel], P, B, seed=seed); t_ks += time.perf_counter() - t1
                t1 = time.perf_counter(); gs = fp.attr_group_shapley(det, XT[sel], G, B); t_gs += time.perf_counter() - t1
                n_expl += len(sel)
                ks_proc = np.stack([phi[:, proc_of == g].sum(axis=1) for g in fp.GROUPS], axis=1)
                tg = e.targets.split("+")
                for mname, A in [("KernelSHAP", ks_proc), ("GroupShapley (same seconds)", gs)]:
                    top = fp.GROUPS[int(np.argmax(A.mean(axis=0)))]
                    rows.append(dict(detector=D.name, seed=seed, method=mname, event=e.event, targets=e.targets,
                                     single=e.single, seconds_explained=len(sel), top=top, hit=float(top in tg)))
            cost.append(dict(detector=D.name, seed=seed, seconds_explained=n_expl,
                             kernel_shap_ms_per_second=1000 * t_ks / max(n_expl, 1),
                             group_shapley_ms_per_second=1000 * t_gs / max(n_expl, 1),
                             kernel_shap_evaluations=KS_SAMPLES + 2, group_shapley_evaluations=16))
            print(f"[{time.time()-t0:6.0f}s] Kernel SHAP {D.name} seed {seed}: {n_expl} seconds, "
                  f"{1000*t_ks/max(n_expl,1):.1f} vs {1000*t_gs/max(n_expl,1):.2f} ms/s", flush=True)
    pd.DataFrame(rows).to_csv(f"{R}/E11_kernelshap_per_event.csv", index=False)
    pd.DataFrame(cost).to_csv(f"{R}/E11_kernelshap_cost.csv", index=False)
    json.dump(dict(usad_window_s=WIN, usad_latent=40, usad_epochs=20, usad_train_windows=60_000, usad_window_pool=200_000,
                   kernel_shap_coalitions=KS_SAMPLES, kernel_shap_seconds_per_event=KS_PER_EVENT,
                   kernel_shap_validation_max_rel_error=val_err, runtime_s=time.time() - t0),
              open(f"{R}/extend_config.json", "w"), indent=2)
    print("extend ok", round(time.time() - t0), "s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cost.py
"""Table 9: computational cost of every detector and attribution, all measured on one machine.
Times are per 1,000 samples; Kernel SHAP uses 1,024 coalitions per explained sample.

Usage:
    python cost.py --data hai/hai-21.03 --results results
"""
import argparse, os, sys, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fdi_pipeline as fp
import extend as ex


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True); ap.add_argument("--results", default="results")
    a = ap.parse_args()
    tr, te = fp.load(a.data)
    feats, gidx, Xfit, Xcal, Xte, base = fp.prepare(tr, te)
    n1, n2 = len(tr[0]), len(tr[1]); del tr
    y = te["attack"].to_numpy(); files = te["file"].to_numpy(); d = Xfit.shape[1]
    att = np.flatnonzero(y == 1)[::4][:1000]
    te_starts = [int(np.flatnonzero(files == f)[0]) for f in np.unique(files)]
    pool = np.sort(np.random.default_rng(0).choice(len(Xfit), size=200_000, replace=False))
    Lfit = ex.lagged(Xfit, [0, n1, n1 + n2], rows=pool); Lcal = ex.lagged(Xcal, [0])
    La = ex.lagged(Xte, te_starts, rows=att); Ls = ex.lagged(Xte, te_starts, rows=np.arange(20000))
    gidx_L = {g: np.concatenate([gidx[g] + k * d for k in range(ex.WIN)]) for g in fp.GROUPS}
    base_L = np.tile(base, ex.WIN).astype(np.float32)
    rows = []
    for D in [fp.PCARecon, fp.MLPAE, fp.IForest, fp.ZMax, ex.USAD]:
        lag = D is ex.USAD
        Xf, Xs, Xa, G, B = (Lfit, Ls, La, gidx_L, base_L) if lag else (Xfit, Xte[:20000], Xte[att], gidx, base)
        players = [np.array([j + k * d for k in range(ex.WIN)]) for j in range(d)] if lag else [np.array([j]) for j in range(d)]
        t = time.perf_counter(); det = D(42).fit(Xf); fit_s = time.perf_counter() - t
        t = time.perf_counter(); det.score(Xs); score = (time.perf_counter() - t) / len(Xs) * 1e6
        r = dict(detector=D.name, fit_s=fit_s, score_ms_per_1k=score)
        t = time.perf_counter(); fp.attr_group_shapley(det, Xa, G, B); r["group_shapley_ms_per_1k"] = (time.perf_counter() - t) * 1000
        occ = lambda X: np.stack([det.score(X) - det.score(np.where(np.isin(np.arange(X.shape[1]), G[g])[None, :], B[None, :], X))
                                  for g in fp.GROUPS], axis=1)
        t = time.perf_counter(); occ(Xa); r["occlusion_ms_per_1k"] = (time.perf_counter() - t) * 1000
        cal_r = det.resid(Xcal[:5000] if not lag else Lcal[:5000])
        if cal_r is not None:
            sd = np.std(cal_r, axis=0) + 1e-6
            t = time.perf_counter(); fp.attr_residual(det, Xa, G, sd); r["residual_ms_per_1k"] = (time.perf_counter() - t) * 1000
        t = time.perf_counter(); fp.attr_z(Xa[:, :d], gidx); r["zmean_ms_per_1k"] = (time.perf_counter() - t) * 1000
        sub = Xa[:100]                                  # Kernel SHAP is timed on 100 samples and scaled to 1,000
        t = time.perf_counter(); ex.kernel_shap(det.score, sub, players, B); r["kernel_shap_ms_per_1k"] = (time.perf_counter() - t) * 1000 * 10
        rows.append(r); print(r, flush=True)
    out = pd.DataFrame(rows); out["cpu"] = platform.processor() or open("/proc/cpuinfo").read().split("model name")[1].split("\n")[0].strip(": ")
    out.to_csv(f"{a.results}/E12_cost.csv", index=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eventmap.py
"""Figure 9: per-event localization map for every detector-attribution combination (incl. USAD).
Usage: python eventmap.py results"""
import sys, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
R = sys.argv[1] if len(sys.argv) > 1 else "results"; F = f"{R}/figures"
plt.rcParams.update({"font.size": 9, "font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED, DARK = "#2a6fd6", "#c0504d", "#555555"
pe = pd.read_csv(f"{R}/E2_localization_per_event.csv"); ev = pd.read_csv(f"{R}/events.csv")
dets = [d for d in ["PCA", "MLP-AE", "IForest", "Z-max", "USAD"] if d in set(pe.detector)]
combos = [(d, m) for d in dets for m in ["Residual", "Z-mean", "GroupShapley"] if not (d == "IForest" and m == "Residual")]
order = ev.sort_values("magnitude_z").event.tolist()
S = np.zeros((len(order), len(combos)))
for j, (d, m) in enumerate(combos):
    sub = pe[(pe.detector == d) & (pe.method == m)]
    for i, e in enumerate(order):
        h = sub[sub.event == e].hit.to_numpy()
        S[i, j] = 0 if np.mean(~np.isnan(h)) < 0.5 else (2 if np.nanmean(h) >= 0.5 else 1)
pd.DataFrame(S, index=order, columns=[f"{d}|{m}" for d, m in combos]).to_csv(f"{R}/E6_event_map.csv")
fig, ax = plt.subplots(figsize=(7.09, 6.4))
ax.imshow(S, cmap=ListedColormap(["#f2f2f2", RED, BLUE]), aspect="auto", vmin=0, vmax=2)
ax.set_xticks(range(len(combos)))
ax.set_xticklabels([f"{d}\n{'G. Shap.' if m == 'GroupShapley' else ('Resid.' if m == 'Residual' else m)}" for d, m in combos], fontsize=5.8)
lab = ev.set_index("event").loc[order]
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{t:<6} {z:6.1f}" for t, z in zip(lab.targets, lab.magnitude_z)], fontsize=5.5, family="monospace")
for b in [17, 33]:
    ax.axhline(b - 0.5, color=DARK, lw=0.8)
x = -0.5
for d in dets[:-1]:
    x += 2 if d == "IForest" else 3
    ax.axvline(x, color="white", lw=2)
for y_, t in [(8, "high\nstealth"), (25, "medium"), (41.5, "low\nstealth")]:
    ax.text(len(combos) - 0.4, y_, t, fontsize=7, va="center")
ax.set_ylabel("Attack event: target process and deviation (z), sorted by deviation", fontsize=7.5)
ax.legend(handles=[Patch(color=BLUE, label="Detected, correct process"), Patch(color=RED, label="Detected, wrong process"),
                   Patch(color="#f2f2f2", label="Not detected")], frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=7.5)
for ext in ("png", "tif"):
    fig.savefig(f"{F}/Fig9_event_map.{ext}", dpi=300, bbox_inches="tight", **({"pil_kwargs": {"compression": "tiff_lzw"}} if ext == "tif" else {}))
print("event map", S.shape)


In [ ]:
%%writefile extend22.py
"""
HAI 22.04 versions of the two additions (run after hai22_pipeline.py and with extend.py next to it):
  H10  USAD through the same detection and multi-level localization protocol (rows detector = "USAD").
  H11  Per-sensor Kernel SHAP summed to processes and control loops vs exact group Shapley,
       on the same sampled attack seconds of every event (oracle setting), all detectors and seeds.

Usage:
    python extend22.py --data hai/hai-22.04 --gt hai22_ground_truth.csv --out results22
"""
import argparse, json, os, sys, time
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fdi_pipeline as fp
import hai22_pipeline as hp
import extend as ex

KS_PER_EVENT = 20


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True); ap.add_argument("--gt", required=True)
    ap.add_argument("--out", default="results22"); ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--ks_only", action="store_true", help="skip H10; retrain USAD only for Kernel SHAP")
    ap.add_argument("--iforest_seeds", type=int, default=1, help="IForest seeds for Kernel SHAP (costly)")
    a = ap.parse_args(); t0 = time.time(); seeds = hp.SEEDS[:a.seeds]

    gt = pd.read_csv(a.gt)
    tr, te = hp.load(a.data)
    feats, Xfit, Xcal, Xte, base, n_train, n_fit, n_cal = hp.prepare(tr, te)
    lens = [len(t) for t in tr[:-1]]; fit_starts = list(np.cumsum([0] + lens))
    del tr
    pidx, lidx = hp.groups_for(feats)
    P_names, L_names = list(pidx), list(lidx)
    Pg, Lg = [pidx[p] for p in P_names], [lidx[L] for L in L_names]
    ev = hp.events(te, gt)
    y = te["Attack"].to_numpy().astype(int); files = te["file"].to_numpy()
    d = Xfit.shape[1]; W = ex.WIN
    rows = np.concatenate([np.arange(e.start, e.end + 1)[::hp.STRIDE] for _, e in ev.iterrows()])
    Xa = Xte[rows]; Z = np.abs(Xa)
    valid_pts = lambda e: [feats.index(p) for p in e.points.split(";") if p in feats]

    # ---------------------------------------------------------------- H10 USAD
    te_starts = [int(np.flatnonzero(files == f)[0]) for f in np.unique(files)]
    pool = np.sort(np.random.default_rng(0).choice(len(Xfit), size=200_000, replace=False))
    Lfit, Lcal, Lte = ex.lagged(Xfit, fit_starts, rows=pool), ex.lagged(Xcal, [0]), ex.lagged(Xte, te_starts)
    lift = lambda g: np.concatenate([g + k * d for k in range(W)])
    PgL, LgL = [lift(g) for g in Pg], [lift(g) for g in Lg]
    baseL = np.tile(base, W).astype(np.float32); La = Lte[rows]
    det_rows, per_ev, usad = [], [], {}
    for seed in seeds:
        det = ex.USAD(seed).fit(Lfit)
        if a.ks_only:
            usad[seed] = det; print(f"[{time.time()-t0:6.0f}s] USAD seed {seed} trained", flush=True); continue
        s_cal = fp.smooth(det.score(Lcal), np.zeros(len(Lcal)))
        s = fp.smooth(det.score(Lte), files); alarm = s > np.quantile(s_cal, fp.THR_Q)
        m, _ = fp.evaluate(alarm, y, files, ev.assign(targets=""), s)
        det_rows.append(dict(detector="USAD", seed=seed, **m)); usad[seed] = det
        cal_sd = np.std(det.resid(Lcal), axis=0) + 1e-6
        RresL = (det.resid(La) / cal_sd) ** 2
        Rres = RresL.reshape(len(La), W, d).mean(axis=1)          # per-sensor residual, averaged over the window
        A = {("process", "GroupShapley"): hp.shapley(det, La, PgL, baseL), ("process", "Z-mean"): hp.group_mean(Z, Pg),
             ("loop", "GroupShapley"): hp.shapley(det, La, LgL, baseL), ("loop", "Occlusion"): hp.occlusion(det, La, LgL, baseL),
             ("loop", "Z-mean"): hp.group_mean(Z, Lg),
             ("process", "Residual"): hp.group_mean(Rres, Pg), ("loop", "Residual"): hp.group_mean(Rres, Lg)}
        for e in ev.itertuples():
            pos = np.flatnonzero((rows >= e.start) & (rows <= e.end)); al = pos[alarm[rows[pos]]]
            for (lvl, meth), M in A.items():
                names = P_names if lvl == "process" else L_names
                targets = e.procs.split(";") if lvl == "process" else e.loops.split(";")
                top_or = names[int(np.argmax(M[pos].mean(axis=0)))]
                top = names[int(np.argmax(M[al].mean(axis=0)))] if len(al) else None
                per_ev.append(dict(detector="USAD", seed=seed, level=lvl, method=meth, id=e.id, detected=int(len(al) > 0),
                                   top=top, hit=(float(top in targets) if top else np.nan), top_oracle=top_or,
                                   hit_oracle=float(top_or in targets)))
            vp = valid_pts(e)
            if vp and not e.ambiguous:
                for meth, M in {"Z-score": Z, "Residual": Rres}.items():
                    for src, P in [("alarm", al), ("oracle", pos)]:
                        if len(P) == 0:
                            continue
                        order = np.argsort(-M[P].mean(axis=0))
                        per_ev.append(dict(detector="USAD", seed=seed, level=f"point-{src}", method=meth, id=e.id,
                                           detected=int(len(al) > 0), top=feats[order[0]], hit=float(order[0] in vp),
                                           hit3=float(bool(set(order[:3]) & set(vp))), hit5=float(bool(set(order[:5]) & set(vp)))))
                for src, P in [("alarm", al), ("oracle", pos)]:
                    if len(P) == 0:
                        continue
                    Lt = L_names[int(np.argmax(A[("loop", "GroupShapley")][P].mean(axis=0)))]
                    inside = lidx[Lt]; order = inside[np.argsort(-Z[P][:, inside].mean(axis=0))]
                    per_ev.append(dict(detector="USAD", seed=seed, level=f"point-{src}", method="Loop Shapley + Z-score",
                                       id=e.id, detected=int(len(al) > 0), top=feats[order[0]], hit=float(order[0] in vp),
                                       hit3=float(bool(set(order[:3]) & set(vp))), hit5=float(bool(set(order[:5]) & set(vp)))))
        print(f"[{time.time()-t0:6.0f}s] USAD seed {seed}: F1={m['f1']:.3f} event recall={m['event_recall']:.2f} "
              f"FA/h={m['false_alarms_per_hour']:.3f}", flush=True)
    if not a.ks_only:
        pd.DataFrame(det_rows).to_csv(f"{a.out}/H10_detection.csv", index=False)
        pd.DataFrame(per_ev).to_csv(f"{a.out}/H10_localization_per_event.csv", index=False)

    # ------------------------------------------------------------ H11 Kernel SHAP
    proc_of = np.array([P_names.index(next(p for p in P_names if j in pidx[p])) for j in range(d)])
    loop_of = np.full(d, -1)
    for li, g in enumerate(Lg):
        loop_of[g] = li
    out = []
    dets = [(fp.PCARecon, seeds[:1]), (fp.MLPAE, seeds), (fp.ZMax, seeds[:1]), ("USAD", seeds), (fp.IForest, seeds[:a.iforest_seeds])]
    for D, sl in dets:
        for seed in sl:
            if D == "USAD":
                det = usad[seed]; XT, B, P_, L_, players = Lte, baseL, PgL, LgL, [np.array([j + k * d for k in range(W)]) for j in range(d)]
                name = "USAD"
            else:
                det = D(seed).fit(Xfit); XT, B, P_, L_, players = Xte, base, Pg, Lg, [np.array([j]) for j in range(d)]
                name = D.name
            for e in ev.itertuples():
                sel = ex.sample_seconds(np.arange(e.start, e.end + 1), KS_PER_EVENT)
                phi = ex.kernel_shap(det.score, XT[sel], players, B, seed=seed)
                gsP = hp.shapley(det, XT[sel], P_, B); gsL = hp.shapley(det, XT[sel], L_, B)
                ksP = np.stack([phi[:, proc_of == i].sum(axis=1) for i in range(len(P_names))], axis=1)
                ksL = np.stack([phi[:, loop_of == i].sum(axis=1) for i in range(len(L_names))], axis=1)
                for lvl, names, targets, pairs in [("process", P_names, e.procs.split(";"), [("KernelSHAP", ksP), ("GroupShapley (same seconds)", gsP)]),
                                                   ("loop", L_names, e.loops.split(";"), [("KernelSHAP", ksL), ("GroupShapley (same seconds)", gsL)])]:
                    for meth, M in pairs:
                        top = names[int(np.argmax(M.mean(axis=0)))]
                        out.append(dict(detector=name, seed=seed, level=lvl, method=meth, id=e.id, top_oracle=top,
                                        hit_oracle=float(top in targets), seconds_explained=len(sel)))
            print(f"[{time.time()-t0:6.0f}s] Kernel SHAP 22.04 {name} seed {seed}", flush=True)
            pd.DataFrame(out).to_csv(f"{a.out}/H11_kernelshap_per_event.csv", index=False)
    pd.DataFrame(out).to_csv(f"{a.out}/H11_kernelshap_per_event.csv", index=False)
    json.dump(dict(runtime_s=time.time() - t0, kernel_shap_seconds_per_event=KS_PER_EVENT,
                   kernel_shap_coalitions=ex.KS_SAMPLES, usad_window_s=W), open(f"{a.out}/extend22.json", "w"), indent=2)
    print("extend22 ok", round(time.time() - t0), "s")


if __name__ == "__main__":
    main()


In [ ]:
!pip -q install torch
# HAI 21.03: USAD through the full protocol + Kernel SHAP (writes E10_*, E11_* into results/)
!python extend.py --data hai/hai-21.03 --results results 2>&1 | grep -v -i warn
# merge USAD rows into the main tables, re-run the analysis and figures
import pandas as pd
for base in ["E1_detection","E2_localization","E2_localization_per_event","E3_stealth_tiers","E4_trust_gate","E4_duration_curve"]:
    a = pd.read_csv(f"results/{base}.csv"); b = pd.read_csv(f"results/E10_{base}.csv").drop(columns="fit_s", errors="ignore")
    pd.concat([a[a.detector != "USAD"], b], ignore_index=True).to_csv(f"results/{base}.csv", index=False)
!python analyze.py results && python eventmap.py results
!python cost.py --data hai/hai-21.03 --results results 2>&1 | grep -v -i warn
# HAI 22.04: USAD + Kernel SHAP (process and control-loop level)
!python extend22.py --data hai/hai-22.04 --gt hai22_ground_truth.csv --out results22 --iforest_seeds 1 2>&1 | grep -v -i warn
for a_, b_ in [("H1_detection","H10_detection"),("H2_localization_per_event","H10_localization_per_event")]:
    a = pd.read_csv(f"results22/{a_}.csv"); b = pd.read_csv(f"results22/{b_}.csv")
    pd.concat([a[a.detector != "USAD"], b], ignore_index=True).to_csv(f"results22/{a_}.csv", index=False)
!python analyze22.py results22 && python diagrams.py --results results --results22 results22
!zip -qr results.zip results results22
try:
    from google.colab import files; files.download('results.zip')
except ImportError:
    print('results.zip created')